# MasterClass: LLM Acceleration at the Edge of Hardware
## A Learning Journey Through Speculative Decoding & FP8 Quantization

> *"The first principle is that you must not fool yourself — and you are the easiest person to fool."*  
> — Richard Feynman

---

This guide is **not** a recipe. It is a journey.

You will not find "run this command and it works" here. Instead, you will find:
- Questions that force you to think before you act
- Small experiments that build intuition
- Theory presented at the moment you need it
- Evidence logs that accumulate into an argument
- A central mystery that resolves only at the end — through your own experiments

**The goal:** Understand *why* these techniques work, *when* they combine well, and *in what order* to apply them — all backed by numbers you generated yourself.

---

## The Setup

| Item | Value |
|---|---|
| Hardware | 1× NVIDIA H100 80GB SXM (Nebius AI Cloud) |
| Model | `Qwen/Qwen3-8B` (BF16, ~16 GB on-device) |
| Goal | Maximize output token throughput on a serving workload |
| Central Question | Which optimization should be applied **first** — and why? |
| Submission | `submission.zip` containing `spec_dec+quantization_homework.ipynb` |

---

## Navigation Map

| Section | What You Will Find |
|---|---|
| **Before Ch. 0 — Infra Selection & Nebius Setup** | **How to size GPU / VRAM / memory bandwidth / CPU / RAM / disk to the workload**, then provision the instance, disk, Python 3.12, model download, pre-flight check |
| **Chapter 0** | The performance puzzle — why is a 312 TFLOPS GPU slow at generation? |
| **Chapter 1** | Two strategies, one bottleneck — map the solution space before coding |
| **Chapter 2** | Speculative decoding — EAGLE-3, offline vs online, disk size math |
| **Chapter 3** | Draft head training — full_acc vs cond_acc, checkpoint selection |
| **Chapter 4** | FP8 quantization — formats, dynamic vs static, lm_head exclusion |
| **Chapter 5** | The Grand Benchmark — design, draft token sweep, all 4 configs |
| **Chapter 6** | The interaction problem — acceptance rate paradox, BF16 vs FP8 distribution |
| **Chapter 7** | The Central Mystery — which goes first, train-on-what-you-serve |
| **Chapter 8** | Evidence synthesis — write your final argument from real data |
| **Chapter 9** | **Filling the submission notebook** — cell-by-cell, questions, zip command |
| **Appendix A** | Quick reference: envs, metrics glossary, troubleshooting table |
| **Appendix B** | Documentation navigation — where to read, when to read it |

---
# Before Chapter 0 — Part A: How to Choose Your Infrastructure

> *Most people pick a cloud instance by copying someone else's spec sheet. Engineers derive it.*

Before you click "Create Instance," learn the skill that outlasts this assignment: **how to size a machine to a workload.** The method is always the same — **work backwards from what the workload actually does to the resource each phase stresses.** Provisioning (Part B) then becomes trivial.

---

## The Golden Rule: Profile the Workload, Then Size Each Resource

No single resource dominates an ML pipeline — different *phases* stress different parts. List the phases, find each one's bottleneck, then size the machine for the **maximum** demand across phases.

| Phase (chapter) | What it does | Dominant resource | Secondary |
|---|---|---|---|
| Baseline serve + benchmark (Ch 0, 5) | Autoregressive decode at batch ≈ 8 | **GPU HBM bandwidth** + VRAM (KV cache) | FP8 tensor cores (for the FP8 configs) |
| Hidden-state generation (Ch 2) | Forward passes over 3000 samples, dump tensors to disk | **GPU compute** (prefill) + **disk capacity** | disk *sequential* bandwidth |
| Draft-head training (Ch 3) | Re-read ~140 GB every epoch for 5 epochs | **GPU compute** + **system RAM** (page cache) | disk *sequential* read bandwidth |
| FP8 quantization (Ch 4) | Load BF16 weights, convert to FP8 | **GPU VRAM** + **host RAM** | — |

**Takeaway:** size GPU for serving + FP8, RAM for training, disk for capacity. Now let's derive each number.

---

## Resource 1 — GPU: Two Numbers, Two Reasons

A GPU spec sheet has many numbers; for LLM inference only two matter, and they matter for *different* phases:

- **Memory bandwidth (HBM)** → the **decode throughput ceiling.** As Chapter 0 proves, every generated token reloads *all* weights from HBM, so `tokens/s ≈ HBM_bandwidth / model_bytes`. The H100's **3.35 TB/s** is the single biggest reason to choose it for a latency/throughput serving task.
- **Compute (TFLOPS)** → matters for **prefill** and **training** (large batched matmuls), not single-stream decode. H100 ≈ **312 TFLOPS** BF16.

Plus one hard constraint:

- **FP8 tensor cores** exist only on **Hopper (H100)** and Ada GPUs. Chapter 4 *requires* native FP8 — an A100 physically cannot run that half of the assignment. **This alone forces the H100 choice.**

> **Decision:** 1× H100 80 GB SXM. (SXM has higher bandwidth than PCIe, but either works for a single-GPU run.)

---

## Resource 2 — VRAM: "Will It Fit?"

Budget the GPU's 80 GB against everything that must live on-device at serving time:

| Lives in VRAM | BF16 | FP8 |
|---|---|---|
| Model weights | ~16 GB | ~8 GB |
| KV cache (concurrency 8, seq ≈ 2k) | a few GB | a few GB |
| EAGLE-3 draft head | < 1 GB | < 1 GB |
| Activations + CUDA + vLLM overhead | a few GB | a few GB |
| **Total** | **~25 GB** | **~17 GB** |

Qwen3-8B fits in far less than 80 GB — so why 80 GB? **Headroom buys options:** higher concurrency, deeper draft trees, larger KV cache, and room for the FP8 + BF16 experiments side-by-side. A 24 GB consumer GPU would *fit* BF16 Qwen3-8B but gives up FP8 support and bandwidth.

**Rule of thumb:** `VRAM ≈ weights + KV cache + ~20% overhead`, where `KV cache ≈ 2 × layers × kv_heads × head_dim × seq_len × batch × bytes`.

---

## Resource 3 — Memory Bandwidth (there are TWO kinds — don't confuse them)

This is the most misunderstood spec, because "bandwidth" appears in two places:

| | What it is | Magnitude | Governs |
|---|---|---|---|
| **GPU HBM bandwidth** | On-package GPU memory | **~3,350,000 MiB/s** (3.35 TB/s) | Decode speed (the roofline ceiling in Ch 0) |
| **Disk bandwidth** | Block-storage throughput | **~285 MiB/s** (your config) | Reading/writing hidden states |

They differ by ~**4 orders of magnitude**. That gap is *why* the model lives in HBM during inference and only touches disk for bulk data movement. When the assignment says "memory-bound," it means **HBM**, not disk.

---

## Resource 4 — System RAM: The Hidden Hero of Training

Draft-head training re-reads the **~140 GB** hidden-state set every epoch (5×). Here's the trick:

- With **enough RAM**, Linux keeps that 140 GB in the **page cache** after epoch 1. Epochs 2–5 then read from RAM (tens of GB/s) instead of disk (0.28 GB/s) — a *massive* speedup, for free.
- FP8 quantization (Ch 4) also loads the BF16 model into **host RAM** before/while moving it to the GPU.

> **Rule:** `RAM ≥ (working set you want cached) + (model-load overhead)`. Here ~**200 GB is ideal** (caches the whole dataset); 128 GB is workable (partial cache); < 64 GB means training is disk-bound every epoch.

---

## Resource 5 — CPU (vCPU): Don't Starve the GPU

The CPU is not the star, but if it's undersized the **GPU sits idle waiting for data**:

- **Dataloader workers** do tokenization and tensor decoding to feed each GPU step.
- Plus model download, filesystem IO, and monitoring.

> **Rule:** ~2–4 vCPU per dataloader worker. **16–32 vCPU** is plenty for a single-GPU run. (More vCPU than that won't help — the GPU is the bottleneck.)

---

## Resource 6 — Disk: Four Independent Knobs

Storage isn't one number — it's **four**, and each maps to a different workload trait. Match each:

**(a) Capacity (GB)** — sum your artifacts (full table in Part B → ~**199 GB**). Choose ≥ 250 GB; 500–600 GB is comfortable. *Your 600 GiB is ample.*

**(b) Bandwidth (MiB/s)** — matters for **large sequential** files. Hidden states are big contiguous tensors, so this is the relevant disk number. Your **285 MiB/s**: writing 140 GB ≈ `140000/285 ≈ 8 min` total, spread across *hours* of GPU work → **not the bottleneck.** First-epoch read ≈ 8 min; later epochs come from RAM cache.

**(c) IOPS** — matters for **small random** operations (many tiny files, metadata, databases). This pipeline is **sequential, not random**, so your **19,000 IOPS** is far more than needed — IOPS is *not* the constraint here.

**(d) Block size (KiB)** — small blocks (4 KiB) favor random IO; large blocks (64–128 KiB) raise *sequential* throughput for big files. For hidden-state dumps a larger block size can help slightly, but the default **4 KiB is fine** given the pipeline is GPU-bound.

> **Cloud gotcha worth remembering:** on Nebius SSD, **performance scales with provisioned size** (e.g., 600 GiB → 285 MiB/s, 1280 GiB → 450 MiB/s). So if disk throughput ever becomes your bottleneck, you have two levers: **grow the disk** (more bandwidth) or **switch tier** (see below).

### Disk Tier Choice (the SSD / SSD NRD / SSD IO toggle)

| Tier | Character | Use when |
|---|---|---|
| **SSD** | Balanced bandwidth + replicated/reliable | **Default — correct for this assignment** |
| **SSD NRD** (non-replicated) | Faster/cheaper, **no redundancy** (data lost on hardware failure) | Only for scratch you can regenerate |
| **SSD IO** | High IOPS + throughput | Random-heavy or throughput-critical IO |

> **For HW3: plain SSD is the right call.** You don't need SSD IO — your bottleneck is the GPU, not the disk.

---

## Single Disk vs Separate Data Disk

The original guide told you to attach a *separate* NVMe disk — but that was only because it assumed a **tiny 100 GB boot disk** that couldn't hold 140 GB of hidden states. **If your boot disk is already large (e.g., 600 GiB), put everything on it** — no second disk, no `mkfs`, no `mount`, one less thing to break. (Part B Step N2-A shows this simpler path.)

---

## The Decision Checklist (reusable for any workload)

| If you see this workload signal... | ...scale this resource |
|---|---|
| Needs FP8 / newest kernels | Hopper-class GPU (H100) |
| Decode throughput / latency target | GPU **HBM bandwidth** + VRAM (KV cache) |
| Big dataset re-read every epoch | **System RAM** (page cache) |
| GPU utilization low, waiting for data | More **vCPU** / dataloader workers |
| Large sequential file IO | Disk **bandwidth** (grow disk size) |
| Many small random files / metadata | Disk **IOPS** (SSD IO tier) |
| Lots of artifacts to store | Disk **capacity** |

---

## This Assignment — The Derived Spec (and your actual setup)

| Resource | Recommended | Driving reason | Your setup | Verdict |
|---|---|---|---|---|
| GPU | 1× H100 80 GB | FP8 + 3.35 TB/s HBM | H100 80 GB SXM | ✅ |
| vCPU | 16–32 | Feed dataloaders | (instance default) | ✅ |
| RAM | 128–200 GB | Cache 140 GB dataset | (instance default) | ✅ if ≥128 GB |
| Disk type | SSD | Balanced + reliable | SSD | ✅ |
| Disk size | ≥ 250 GB (500–600 comfy) | Fit ~199 GB artifacts | **600 GiB** | ✅ ample |
| Disk bandwidth | ~285 MiB/s is fine | Sequential dumps; GPU-bound anyway | **285 MiB/s** | ✅ |
| Disk IOPS | not critical | Sequential, not random | **19,000** | ✅ overkill |
| OS | Ubuntu 22.04 + CUDA 12 | FP8/vLLM stack | Ubuntu 22.04 CUDA 12 | ✅ |

> **Verdict for your configuration:** well-matched. A **single 600 GiB SSD boot disk is sufficient** — you do **not** need a separate NVMe data disk, and the standard SSD tier is correct.

> **Curiosity Probe:** If you doubled the dataset to 6000 samples (~280 GB of hidden states), which resource breaks *first*? Think it through before Part B. *(Hint: capacity goes first; then RAM can no longer cache the whole set, so training becomes disk-bandwidth-bound on every epoch.)*

---
# Before Chapter 0 — Part B: Provisioning the Instance on Nebius

Part A taught you *how to reason* about hardware. Now apply it. Below is the concrete Nebius click-path, with the **reasoning attached to every choice** so you can adapt it to any future workload — not just copy a spec sheet.

---

## Step N1: Provision the Instance

In the Nebius console: **Compute → Create Instance**, then choose:

| Setting | Value | Why this value (derived in Part A) |
|---|---|---|
| GPU | 1× H100 80GB SXM | FP8 tensor cores are **required** for Chapter 4 (only Hopper/Ada have them) + 3.35 TB/s HBM is the decode-throughput ceiling |
| vCPU | 16–32 | Feed the dataloader workers so the GPU never starves during training |
| RAM | 128–200 GB | Page-cache the ~140 GB hidden-state set so training epochs 2–5 read from RAM, not disk |
| Disk type | **SSD** | Balanced bandwidth + reliability — the correct default (not NRD/IO) for this workload |
| Disk size | 600 GB (single boot disk) | Holds the full ~199 GB of artifacts with comfortable headroom — **no second disk needed** |
| OS image | Ubuntu 22.04 LTS for NVIDIA GPUs (CUDA 12) | Ships the CUDA 12 stack that vLLM + FP8 require, pre-installed |

> **Two valid storage layouts — pick one:**
> - **(A) Single large boot disk** *(recommended — and what we use here):* if your boot disk is ≥ 250 GB, put everything on it. Simplest, one less failure mode. → use **Step N2-A**.
> - **(B) Small boot disk + separate data disk:** only if your boot disk is tiny (e.g., 100 GB) and you attached a second NVMe. → use **Step N2-B**.

> **Why we dropped the old "always attach a separate disk" advice:** that was only needed because the original spec assumed a 100 GB boot disk too small for 140 GB of hidden states. With a 600 GB boot disk, the capacity problem disappears.

---

## Step N2-A: Prepare the Workspace (single boot disk — recommended)

Your boot disk already has room, so there is **nothing to format or mount**. `/data` is just a directory on the disk you already have. Create the project layout:

```bash
# /data is just a folder on your boot disk (NO mkfs, NO mount needed)
sudo mkdir -p /data
sudo chown $USER:$USER /data
mkdir -p /data/hw3/{hidden_states,output/checkpoints,Qwen3-8B-FP8-Dynamic}
cd /data/hw3
```

> ⚠️ **Do NOT run `mkfs.ext4` in this layout.** You have only one device (the boot disk); formatting it destroys your OS. `mkfs`/`mount` belong to Step N2-B only.

---

## Step N2-B: Mount a Separate Data Disk (ONLY if you attached one)

Use this **only** for layout (B). These commands turn a raw, empty attached disk into a usable, reboot-persistent `/data`:

```bash
# Find the attached disk (usually /dev/vdb or /dev/nvme1n1)
lsblk

# Format — ONLY a brand-new, EMPTY disk (DESTROYS existing data!)
sudo mkfs.ext4 /dev/vdb

# Mount and take ownership
sudo mkdir -p /data
sudo mount /dev/vdb /data
sudo chown $USER:$USER /data

# Persist across reboots — without this, /data vanishes on restart
echo '/dev/vdb /data ext4 defaults 0 2' | sudo tee -a /etc/fstab

# Create project layout
mkdir -p /data/hw3/{hidden_states,output/checkpoints,Qwen3-8B-FP8-Dynamic}
cd /data/hw3
```

---

## Step N3: Install Python 3.12 and Base System Tools

```bash
# python3.10-venv — needed for speculators_venv (system python3 is 3.10)
sudo apt update && sudo apt install -y python3.10-venv

# Python 3.12 — needed for vllm_venv and comp_venv
sudo apt install -y software-properties-common
sudo add-apt-repository ppa:deadsnakes/ppa -y
sudo apt install -y python3.12 python3.12-venv python3.12-dev

# Utility tools
sudo apt install -y git tmux htop nvtop

# Verify CUDA is visible
nvidia-smi
nvcc --version
```

---

## Step N4: Set Up the Three Python Environments

Three isolated venvs are required because `speculators`, `vllm`, and `llmcompressor` have conflicting dependencies:

| venv | Purpose | Key packages | Needed from |
|---|---|---|---|
| `vllm_venv` | Serving + benchmarking (all 4 configs) | `vllm==0.20.0` | Chapter 0 |
| `speculators_venv` | Hidden state generation + EAGLE-3 training | `speculators v0.5.0` | Chapter 2 |
| `comp_venv` | FP8 quantization | `llmcompressor==0.12.0` | Chapter 4 |

```bash
# ---- vllm_venv (Python 3.12) ----
python3.12 -m venv /data/hw3/vllm_venv
source /data/hw3/vllm_venv/bin/activate
pip install vllm==0.20.0 'fastapi<0.137' datasets
deactivate

# ---- speculators_venv ----
python3 -m venv /data/hw3/speculators_venv
source /data/hw3/speculators_venv/bin/activate
git clone https://github.com/vllm-project/speculators.git /data/hw3/speculators
cd /data/hw3/speculators && git checkout v0.5.0
pip install .
cd /data/hw3
deactivate

# ---- comp_venv (Python 3.12) ----
python3.12 -m venv /data/hw3/comp_venv
source /data/hw3/comp_venv/bin/activate
pip install llmcompressor==0.12.0
deactivate
```

> **Rule:** Never activate two venvs simultaneously. Each chapter header tells you which one to activate.

---

## Step N5: Download Qwen3-8B

```bash
# huggingface_hub is pre-installed in the venvs — activate vllm_venv first
source /data/hw3/vllm_venv/bin/activate

# Authenticate (token from huggingface.co/settings/tokens)
hf auth login

# Download to the data directory
hf download Qwen/Qwen3-8B --local-dir /data/hw3/Qwen3-8B

# Verify (~16 GB expected)
du -sh /data/hw3/Qwen3-8B/
```

---

## Step N6: Use tmux for Long-Running Jobs

Hidden state generation runs for several hours. SSH will disconnect. Use tmux:

```bash
# New named session
tmux new-session -s hw3

# Split into two panes (Ctrl+B then %)
# Switch panes: Ctrl+B then arrow key
# Detach (keeps running): Ctrl+B then D
# Reattach later: tmux attach -t hw3

# Recommended pane layout:
# Pane 1: vLLM server
# Pane 2: hidden state generator / training command
# Pane 3: disk + GPU monitor (watch -n 5 'df -h /data && nvidia-smi')
```

---

## Step N7: Pre-Flight Disk Budget Audit

Before starting, confirm you have enough space. This is the capacity math from Part A, itemized:

```
Component                      | Size estimate
-------------------------------|---------------
Qwen3-8B (BF16 original)       | ~16 GB
Qwen3-8B-FP8-Dynamic           | ~8 GB
speculators_venv               | ~5 GB
vllm_venv                      | ~8 GB
comp_venv                      | ~4 GB
hidden_states/ (3000 samples)  | ~140 GB
output/checkpoints/            | ~2 GB
HF model cache (/data/hf_cache)| ~16 GB
-------------------------------|---------------
TOTAL                          | ~199 GB
Your disk (single 600 GB SSD)  | ✅ ample headroom (~400 GB free)
Minimum viable                 | 250 GB (reduce samples to ~1500)
```

**Stop rule:** If `/data` shows < 50 GB free during generation, pause and reduce `--max-samples` before continuing.

---

## Step N8: Optional — Back Up to Nebius Object Storage

Nebius provides S3-compatible storage. Back up the quantized model and best checkpoint so you can restore them if the instance stops:

```bash
# Configure with Nebius credentials (console → Object Storage → Access Keys)
pip install awscli
aws configure --profile nebius
# Region: eu-north1  (adjust for your region)
# Endpoint: https://storage.eu-north1.nebius.cloud

# Upload quantized model
aws s3 cp /data/hw3/Qwen3-8B-FP8-Dynamic/ \
    s3://your-bucket/hw3/Qwen3-8B-FP8-Dynamic/ \
    --recursive --profile nebius \
    --endpoint-url https://storage.eu-north1.nebius.cloud

# Upload best checkpoint
aws s3 cp /data/hw3/output/checkpoints/ \
    s3://your-bucket/hw3/checkpoints/ \
    --recursive --profile nebius \
    --endpoint-url https://storage.eu-north1.nebius.cloud
```


---

## Step N9: Create a Disk Image for Full Environment Recovery

> **When to do this:** Right after hidden state generation completes and before training starts — that is the most expensive checkpoint to recreate. A disk image captures your entire environment (OS, CUDA, all three venvs, the model, and the hidden states) so a new VM can boot into the same state within minutes.

**Why two layers of protection:**

| Method | What it protects | Restore time |
|---|---|---|
| **Disk image** (this step) | Full environment — OS, CUDA, venvs, model, data | Minutes (boot new VM) |
| **Object Storage** (Step N8) | Critical data files only — hidden states + checkpoints | ~hours (re-download + re-setup) |

Do both. The disk image is fast to restore; Object Storage is cheaper for long-term retention.

---

### Prerequisites — Configure the Nebius CLI

```bash
# Install if not already present
pip install nebius

# Authenticate (opens browser)
nebius profile create
```

---

### Create the Image

```bash
# 1. Stop the instance for a consistent filesystem snapshot
#    (do this from the Nebius console or via CLI)
nebius compute instance stop --id <your-instance-id>

# 2. Find your boot disk ID
nebius compute disk list --parent-id <your-project-id>

# 3. Create a bootable image from the disk
nebius compute image create \
    --name hw3-post-datagen \
    --source-disk-id <boot-disk-id> \
    --cpu-architecture amd64 \
    --parent-id <your-project-id>

# 4. Note the image ID from the output, then restart your instance
nebius compute instance start --id <your-instance-id>
```

---

### Restore on a New VM

If your instance is killed, create a replacement with:

```bash
nebius compute instance create \
    --name hw3-restored \
    --boot-disk-managed-disk-source-image-id <image-id> \
    --boot-disk-managed-disk-size-gibibytes 600 \
    --boot-disk-managed-disk-type network_ssd \
    --parent-id <your-project-id>
    # add --platform, --gpu-cluster-id etc. as per Step N1
```

The new VM boots into the exact state captured in the image — no reinstallation needed.

---

> **Note on Disk Snapshots:** Nebius Aether 3.6 (July 2026) is adding native disk snapshots — live point-in-time copies without stopping the VM. Once available, `nebius compute snapshot create` will be the lower-friction alternative to a full image. Until then, the image approach above is the reliable path.


In [ ]:
# === NEBIUS PRE-FLIGHT CHECK ===
# Save this as nebius_preflight.sh on your Nebius instance and run: bash nebius_preflight.sh

preflight = """
#!/bin/bash
set -e

echo '============================================='
echo ' Nebius AI Cloud — HW3 Pre-Flight Checklist'
echo '============================================='

echo
echo '--- GPU ---'
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader \
  && echo '[OK] H100 detected' || echo '[FAIL] No GPU found'

echo
echo '--- GPU memory bandwidth note ---'
echo '[INFO] H100 HBM ~3.35 TB/s = the decode throughput ceiling (see Chapter 0)'

echo
echo '--- CUDA ---'
nvcc --version 2>/dev/null | head -1 || echo '[WARN] nvcc not in PATH (may be OK)'

echo
echo '--- Python 3.12 ---'
python3.12 --version && echo '[OK]' || echo '[FAIL] Install Python 3.12 first'

echo
echo '--- System RAM (want >=128 GB to cache hidden states) ---'
RAM_GB=$(free -g | awk '/^Mem:/{print $2}')
echo "Total RAM: ${RAM_GB} GB"
[ "${RAM_GB:-0}" -ge 120 ] && echo '[OK] Enough RAM to page-cache the dataset' || echo '[WARN] <128GB: training epochs will re-read from disk'

echo
echo '--- Workspace (/data — folder on boot disk OR mounted data disk) ---'
df -h /data 2>/dev/null && echo '[OK] /data exists' || echo '[INFO] /data missing — create it (Step N2-A)'

echo
echo '--- Project directory ---'
ls /data/hw3/ 2>/dev/null && echo '[OK]' || echo '[WARN] /data/hw3 not created yet'

echo
echo '--- Model download ---'
MODEL_SIZE=$(du -sh /data/hw3/Qwen3-8B/ 2>/dev/null | cut -f1)
[ -n "$MODEL_SIZE" ] && echo "[OK] Qwen3-8B: $MODEL_SIZE" || echo '[WARN] Model not downloaded yet'

echo
echo '--- Free disk ---'
FREE_GB=$(df /data --output=avail -BG 2>/dev/null | tail -1 | tr -d 'G ')
echo "Free space on /data: ${FREE_GB} GB"
[ "${FREE_GB:-0}" -gt 200 ] && echo '[OK] Enough space' || echo '[WARN] Less than 200GB free — consider reducing samples'

echo
echo '--- tmux ---'
tmux -V && echo '[OK]' || echo '[WARN] Install tmux: sudo apt install tmux'

echo
echo '============================================='
echo 'If all checks pass, proceed to Chapter 2.'
echo '============================================='
"""

print(preflight)
print()
print("To run on Nebius:")
print("  1. SSH into your instance")  
print("  2. Save the script: nano nebius_preflight.sh")
print("  3. Paste the content and save (Ctrl+O, Enter, Ctrl+X)")
print("  4. Run: bash nebius_preflight.sh")

---
# Chapter 0 — The Performance Puzzle
## Why Is a 312 TFLOPS GPU Slow at LLM Generation?

Before touching any code, ask yourself a simple question:

> **If the H100 can do 312 TFLOPS of BF16 matrix math, and Qwen3-8B has only 8 billion parameters, why can't it generate thousands of tokens per second?**

Sit with that for a moment. Do the back-of-envelope math.

---

### The Arithmetic

A single forward pass of an 8B transformer with `batch_size=1` costs roughly **16 GFLOPs** (2 × 8B multiply-adds).

Peak H100 BF16 throughput: **312 TFLOPS = 312,000 GFLOPs/s**

Theoretical max tokens/s (compute-bound): `312,000 / 16 ≈ 19,500 tokens/s`

But what do we actually observe in the baseline?  
**~840 tokens/s** — about **23× slower** than theoretical peak.

> **Experiment 0.1:** Run the baseline benchmark now, then come back and fill in the calculator below.

**Environment:** `vllm_venv` | **Working dir:** `/data/hw3`

```bash
# ── Terminal 1: start the model server ──
source /data/hw3/vllm_venv/bin/activate
vllm serve /data/hw3/Qwen3-8B --no-enable-prefix-caching
# Wait for: "Application startup complete."
```

```bash
# ── Terminal 2: run the benchmark (keep Terminal 1 running) ──
source /data/hw3/vllm_venv/bin/activate
vllm bench serve \
    --model /data/hw3/Qwen3-8B \
    --dataset-name hf \
    --dataset-path philschmid/mt-bench \
    --max-concurrency 8 \
    --num-prompts 80
```

From the output, record:
- `Output token throughput (tok/s)` → `baseline_output_toks_per_sec`
- `Mean TPOT (ms)` → `baseline_mean_tpot_ms`
- `Mean TTFT (ms)` → for Evidence Log Entry #0

Then compute:
```
measured_toks_per_sec = 1000 / mean_tpot_ms
utilization = measured_toks_per_sec / 19500
```
What fraction of compute are you using? Write it in the evidence log below.

---

### The Real Bottleneck: Memory Bandwidth

Each token generation step requires **loading all 8B parameters from HBM** (High Bandwidth Memory).  
In BF16, that is: `8B × 2 bytes = 16 GB` of data moved per step.

H100 HBM bandwidth: **3.35 TB/s**

Theoretical max tokens/s (bandwidth-bound): `3350 GB/s / 16 GB = 209 tokens/s per stream`

But we have batch effects and KV cache overhead — vLLM batches requests together, which increases the effective arithmetic intensity. That's why we see ~840 tok/s rather than 209.

> **Key Insight:** LLM token generation at low batch sizes is **memory bandwidth-bound**, not compute-bound. The GPU's compute units sit idle waiting for data to arrive from HBM.

This is the **Roofline Model** insight, and it defines exactly which optimizations can help:

| Optimization | What It Addresses |
|---|---|
| Speculative Decoding | Reduces the *number of memory loads per useful token* (verify multiple tokens per sweep) |
| FP8 Quantization | Reduces *bytes per parameter* (16 GB → 8 GB per sweep) |
| Batching / Continuous Batching | Increases arithmetic intensity per memory load |

Both approaches in this assignment attack the **memory wall** — but from different angles.


In [ ]:
# === EXPERIMENT 0.1: Roofline Calculator ===
# Fill in your measured values after running the baseline benchmark.
# This is your first entry in the evidence log.

# Hardware constants
H100_BF16_TFLOPS = 312_000   # GFLOPs/s
H100_HBM_BANDWIDTH_GBs = 3350  # GB/s

# Model constants
model_params = 8e9
bytes_per_param_bf16 = 2
model_size_gb = model_params * bytes_per_param_bf16 / 1e9

# Theoretical bounds
flops_per_token = 2 * model_params / 1e9  # GFLOPs
compute_bound_toks = H100_BF16_TFLOPS / flops_per_token
bandwidth_bound_toks = H100_HBM_BANDWIDTH_GBs / model_size_gb

print(f"Model size (BF16):             {model_size_gb:.1f} GB")
print(f"FLOPs per token:               {flops_per_token:.1f} GFLOPs")
print(f"Compute-bound ceiling:         {compute_bound_toks:.0f} tok/s")
print(f"Bandwidth-bound ceiling (b=1): {bandwidth_bound_toks:.0f} tok/s")

# ==== FILL IN FROM YOUR BASELINE RUN ====
baseline_output_toks_per_sec = 837.76   # e.g. 841.22
baseline_mean_tpot_ms = 7.37         # e.g. 7.28
# =========================================

if baseline_output_toks_per_sec:
    utilization = baseline_output_toks_per_sec / compute_bound_toks
    bandwidth_efficiency = baseline_output_toks_per_sec / bandwidth_bound_toks
    print(f"\n--- YOUR BASELINE ---")
    print(f"Measured:                      {baseline_output_toks_per_sec:.1f} tok/s")
    print(f"Compute utilization:           {utilization*100:.2f}%")
    print(f"Bandwidth efficiency:          {bandwidth_efficiency*100:.1f}% of single-stream ceiling")
    print(f"\nBottleneck diagnosis:          {'BANDWIDTH' if utilization < 0.05 else 'COMPUTE'}")

### Evidence Log Entry #0

```
DATE: 30/06/2026
CONFIG: Baseline Qwen3-8B (BF16, vLLM)
BENCHMARK: vllm bench serve --max-concurrency 8 --num-prompts 80

Output tok/s:    837.76
Mean TPOT (ms):  7.37
Mean TTFT (ms):  563
Compute util%:   0.043

HYPOTHESIS: Is this run compute-bound or bandwidth-bound?
ANSWER: bandwidth-bound
```

> **Curiosity Probe:** TTFT (time to first token) is much larger than TPOT (time per output token). Why? What does TTFT represent computationally that TPOT does not?


---
# Chapter 1 — Two Strategies, One Bottleneck

Now that you know the bottleneck, let's map out the two strategies before touching any code.

## Strategy A: Do More Work Per Memory Load (Speculative Decoding)

Normal autoregressive generation:
```
Step 1: Load 16GB → Generate token t+1
Step 2: Load 16GB → Generate token t+2
Step 3: Load 16GB → Generate token t+3
Total: 3 × 16GB = 48 GB moved, 3 tokens produced
```

Speculative decoding with a draft head:
```
Step 1: Draft head (tiny, ~free) proposes tokens [t+1, t+2, t+3]
Step 2: Load 16GB → Verify all 3 draft tokens in ONE pass
        (if they're right, accept them all; if wrong, correct and continue)
Total: ~1.1 × 16GB = 17.6 GB moved, up to 3 tokens produced
```

The key: **batched verification is essentially free** compared to sequential generation.

## Strategy B: Reduce Data Per Load (FP8 Quantization)

BF16 parameters: `8B × 2 bytes = 16 GB`  
FP8 parameters: `8B × 1 byte = 8 GB`

```
BF16 Step: Load 16GB → Generate token t+1  →  ~4.8ms at peak bandwidth
FP8 Step:  Load  8GB → Generate token t+1  →  ~2.4ms at peak bandwidth
```

FP8 halves the memory load per step — but also requires the H100's FP8 tensor cores.

---

> **Curiosity Probe:** Before running any code, predict which strategy should give larger throughput gains for this workload (batch_size ≈ 8). Write your prediction and reasoning here, then compare after experiments.

```
MY PREDICTION:
Strategy A (Spec Decoding) gain estimate: 1.8 × speedup
Strategy B (FP8 Quant) gain estimate:     2 × speedup
Combined gain estimate:                   3.6 × speedup
Reasoning: Speculative decoding reduces the number of generation steps by leveraging a smaller draft model to predict multiple tokens at once, which can significantly reduce the total computation time. FP8 quantization reduces the memory bandwidth requirements by halving the data size, which can lead to faster memory access and potentially higher throughput. The combination of both strategies should provide a substantial speedup.

FP8 quantization of Model Parameters ensures that the memory bandwidth is reduced by half, which can lead to faster memory access and potentially higher throughput. Speculative Decoding speeedup depends on the accuracy of the draft model and the number of tokens generated per step, which in this case can range from 1 to 3 tokens per step. If only 1 token is approved out of 3 the speedup won't be as high and thus the speedup may avgerage to be 1.8 ×. But combined with FP8 quantization, the speedup should still be significant around 3.6 ×.
```


---
# Chapter 2 — Speculative Decoding: Predicting the Predictor

## 2.1 The Core Idea

Speculative decoding was introduced by [Leviathan et al., 2023](https://arxiv.org/abs/2211.17192) and simultaneously by [Chen et al., 2023](https://arxiv.org/abs/2302.01318). The insight is elegant:

**The verifier (large model) and a draft model agree on tokens most of the time — especially for common continuations.**

If a small, cheap draft model can correctly predict the next few tokens 40–60% of the time, we can verify multiple predictions in a single forward pass of the large model — because Transformer attention is **parallel over the sequence dimension**.

```
[Prompt] → [Draft model proposes: "the cat sat"] → [Verifier checks all 3 at once]
                                                      → accepts "the cat" (2 tokens)
                                                      → rejects "sat", generates "lay"
Result: 3 draft tokens proposed, 2 accepted + 1 corrected = 3 useful tokens in 1 verifier pass
```

The acceptance is done via **rejection sampling** — preserving the exact distribution of the verifier model. The output distribution is mathematically identical to the original model. This is not an approximation.

---

## 2.2 Why EAGLE? Why EAGLE-3?

Standard speculative decoding uses a separate smaller model (e.g. Qwen3-1B as draft for Qwen3-8B). Problems:
- Separate model = memory for two models
- Alignment between draft and verifier is imperfect
- Training from scratch is expensive

**EAGLE** (Extrapolation Algorithm for Greater Language-model Efficiency) takes a different approach:

> Instead of training a separate model, train a **lightweight draft head** that operates on the **hidden states** of the verifier's last layer.

The draft head doesn't process raw tokens — it sees the rich internal representation of the verifier. This makes it much easier to predict the next hidden state (and thus the next token) accurately.

**EAGLE-3** extends this further:
- Uses **multi-layer hidden states** (not just the last layer) as input to the draft head
- Better captures information from different abstraction levels
- Achieves higher acceptance rates with fewer draft tokens

---

> **Curiosity Probe:** Why would using hidden states from multiple layers (EAGLE-3) be better than just the last layer (EAGLE)? Think about what different layers in a transformer encode.


---
## 2.3 Offline vs Online Training — An Important Trade-off

Training the draft head requires hidden states from the verifier. There are two ways to get them:

### Online Training
```
GPU_1 (Inference):  [Verifier model] → hidden states → GPU_2 every batch
GPU_2 (Training):   [Draft head]     ← hidden states ← GPU_1 every batch
```
- Faster end-to-end
- Requires 2 GPUs simultaneously
- Hidden states are never stored to disk

### Offline Training
```
Phase 1 (Inference): [Verifier model] → hidden states → DISK (100+ GB)
Phase 2 (Training):  [Draft head]     ← hidden states ← DISK
```
- Sequential on a single GPU (our constraint: 1× H100)
- Requires enormous disk space
- Verifier can be unloaded during training

---

### Why Do Hidden States Take 140+ GB for 3000 Samples?

> **Experiment 2.1:** Before running hidden state generation, estimate the disk size yourself.

For Qwen3-8B:
- Hidden dimension: `4096`
- Number of layers: `36`
- Sequence length: `2048` tokens
- Batch size: 1 sample
- EAGLE-3 uses hidden states from **multiple layers** (let's say L layers)
- Data type: BF16 (2 bytes)


In [ ]:
# === EXPERIMENT 2.1: Hidden State Disk Space Calculator ===

hidden_dim = 4096          # Qwen3-8B hidden dimension
num_layers = 36            # Transformer layers
seq_len = 2048             # Max sequence length
bytes_per_value = 2        # BF16
num_samples = 3000

# EAGLE-3 uses hidden states from multiple layers — let's explore
for layers_used in [1, 3, 36]:
    bytes_per_sample = hidden_dim * seq_len * layers_used * bytes_per_value
    total_gb = bytes_per_sample * num_samples / 1e9
    print(f"Layers used: {layers_used:2d} | Per sample: {bytes_per_sample/1e6:.1f} MB | "
          f"Total ({num_samples} samples): {total_gb:.1f} GB")

print()
print("Compare: Original text dataset (3000 conversations):")
avg_tokens = 2048
avg_chars_per_token = 4
text_gb = num_samples * avg_tokens * avg_chars_per_token / 1e9
print(f"  Approx text size: {text_gb*1000:.1f} MB")
print()
print("Ratio (hidden states vs text):")
for layers_used in [1, 3, 36]:
    bytes_per_sample = hidden_dim * seq_len * layers_used * bytes_per_value
    total_gb = bytes_per_sample * num_samples / 1e9
    print(f"  {layers_used} layers: {total_gb / text_gb:.0f}× larger than text")

# =============================================================
# === LIVE DISK ASSESSMENT (runs against this machine) ===
# =============================================================
import shutil, os

print()
print("=" * 60)
print("  LIVE DISK ASSESSMENT")
print("=" * 60)

DATA_DIR = "/data/hw3"

# --- 1. Disk totals ---
total, used, free = shutil.disk_usage(DATA_DIR if os.path.exists(DATA_DIR) else "/")
print(f"\nDisk (mounted at {DATA_DIR if os.path.exists(DATA_DIR) else '/'}):")
print(f"  Total : {total/1e9:>7.1f} GB")
print(f"  Used  : {used/1e9:>7.1f} GB")
print(f"  Free  : {free/1e9:>7.1f} GB")

# --- 2. Current /data/hw3 breakdown ---
items = {
    "Qwen3-8B (model)"         : f"{DATA_DIR}/Qwen3-8B",
    "Qwen3-8B-FP8-Dynamic"     : f"{DATA_DIR}/Qwen3-8B-FP8-Dynamic",
    "vllm_venv"                : f"{DATA_DIR}/vllm_venv",
    "speculators_venv"         : f"{DATA_DIR}/speculators_venv",
    "comp_venv"                : f"{DATA_DIR}/comp_venv",
    "hidden_states/"           : f"{DATA_DIR}/hidden_states",
    "output/checkpoints/"      : f"{DATA_DIR}/output",
}

def dir_size_gb(path):
    if not os.path.exists(path):
        return None
    total_bytes = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            try:
                total_bytes += os.path.getsize(os.path.join(dirpath, f))
            except OSError:
                pass
    return total_bytes / 1e9

print(f"\n{'Component':<28} {'On disk':>9}  {'Expected':>9}  Status")
print("-" * 62)
expected = {
    "Qwen3-8B (model)"         : 16.0,
    "Qwen3-8B-FP8-Dynamic"     : 8.0,
    "vllm_venv"                : 8.0,
    "speculators_venv"         : 5.0,
    "comp_venv"                : 4.0,
    "hidden_states/"           : 140.0,
    "output/checkpoints/"      : 2.0,
}

already_used_gb = 0.0
remaining_gb    = 0.0

for label, path in items.items():
    exp = expected[label]
    size = dir_size_gb(path)
    if size is None:
        status = "NOT FOUND"
        on_disk_str = "—"
        remaining_gb += exp
    elif size < 0.01:
        status = "pending"
        on_disk_str = f"{size*1000:.0f} KB"
        remaining_gb += exp
    else:
        status = "✓ done"
        on_disk_str = f"{size:.1f} GB"
        already_used_gb += size
    print(f"  {label:<26} {on_disk_str:>9}  {exp:>7.1f} GB  {status}")

# --- 3. Sufficiency verdict ---
print("-" * 62)
print(f"  {'Already written':<26} {already_used_gb:>7.1f} GB")
print(f"  {'Still to write':<26} {remaining_gb:>7.1f} GB")
print(f"  {'Free on disk':<26} {free/1e9:>7.1f} GB")
print()

headroom = free/1e9 - remaining_gb
if headroom > 50:
    verdict = f"✅  SUFFICIENT  —  {headroom:.0f} GB headroom after all remaining writes"
elif headroom > 0:
    verdict = f"⚠️   TIGHT  —  only {headroom:.0f} GB headroom; reduce --max-samples if needed"
else:
    verdict = f"❌  INSUFFICIENT  —  {abs(headroom):.0f} GB short; must reduce --max-samples"

print(f"  Verdict: {verdict}")
print()
print(f"  Stop rule: pause if free space drops below 50 GB during generation.")


**Key insight from Experiment 2.1:**

Hidden states are enormous because they are **dense floating-point tensors** at every position in the sequence. Text is compressed by tokenization — each token represents ~4 characters but is stored as a single integer (2–4 bytes). Hidden states represent each token as a `4096-dimensional float vector`, which is ~8192 bytes per token position × layers used.

```
Text:             "Hello world" → [9906, 1917]  → 4 bytes
Hidden states:    token 9906   → [0.12, -0.45, ..., 0.33]  → 4096 × 2 bytes = 8192 bytes
Compression ratio: ~2000× more data in hidden states than text
```

This is why **disk monitoring is critical** during data generation.

---

> **Curiosity Probe:** If disk space is the bottleneck, what would you sacrifice to fit in available disk? More samples with shorter sequences, or fewer samples with full-length sequences? Does sample diversity or sequence length matter more for a draft head?


---
## 2.4 Environment Setup — The Dependency Problem

One of the first real obstacles: `speculators` and `vllm` v0.20.0 have conflicting dependencies. This is not a bug — it reflects the fast-moving nature of the LLM tooling ecosystem.

The solution is **isolated virtual environments**, each with its own purpose:

| Environment | Purpose | Key Packages |
|---|---|---|
| `speculators_venv` | Data prep + draft head training | `speculators==v0.5.0` (editable) |
| `vllm_venv` | Serving + benchmarking | `vllm==0.20.0`, `fastapi<0.137` |
| `comp_venv` | FP8 quantization | `llmcompressor==0.12.0` |

---

### Pre-flight Check — Verify Environments (created in Step N4)

In [ ]:
# === 2.4 VENV READINESS CHECK ===
# These environments were created in Step N4 (Before Chapter 0).
# Run this cell to confirm all three are present and functional before proceeding.

import subprocess, os

BASE = "/data/hw3"

checks = [
    {
        "name"    : "speculators_venv",
        "python"  : f"{BASE}/speculators_venv/bin/python",
        "pkg"     : "speculators",
        "purpose" : "hidden state generation + EAGLE-3 training",
    },
    {
        "name"    : "vllm_venv",
        "python"  : f"{BASE}/vllm_venv/bin/python",
        "pkg"     : "vllm",
        "purpose" : "serving + benchmarking",
    },
    {
        "name"    : "comp_venv",
        "python"  : f"{BASE}/comp_venv/bin/python",
        "pkg"     : "llmcompressor",
        "purpose" : "FP8 quantization",
    },
]

VERSION_CMD = "import importlib.metadata; print(importlib.metadata.version({pkg!r}))"

print(f"{'Environment':<22} {'Purpose':<38} {'Status'}")
print("-" * 80)

all_ok = True
for c in checks:
    if not os.path.exists(c["python"]):
        status = "❌  NOT FOUND — run Step N4 to create it"
        all_ok = False
    else:
        result = subprocess.run(
            [c["python"], "-c", VERSION_CMD.format(pkg=c["pkg"])],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            version = result.stdout.strip()
            status = f"✅  ready  (v{version})"
        else:
            status = f"❌  package missing — {result.stderr.strip()[:60]}"
            all_ok = False
    print(f"  {c['name']:<20} {c['purpose']:<38} {status}")

print()
if all_ok:
    print("All environments ready. Proceed to Section 2.5.")
else:
    print("⚠️  One or more environments missing. Return to Step N4 and run the setup commands.")


---
## 2.5 Data Preparation — What Does EAGLE-3 Actually Learn?

The draft head learns to predict the verifier's next hidden state given the current hidden states. The training data is the verifier's own behavior on real conversations.

**Why ShareGPT?** It provides diverse, multi-turn conversational data — the same distribution as typical LLM serving workloads. A draft head trained on diverse data generalizes better to real serving traffic.

> **Curiosity Probe:** What would happen if you trained the draft head on code-only data but served it on general conversations? Would acceptance rate drop? Why?

### Step 1: Write & Run the Data Preparation Script

In [ ]:
# === STEP 1: Write the data preparation script, then run it in a terminal ===
# prepare_data.py tokenizes ShareGPT and must complete before hidden state generation.
import os, stat

SCRIPTS_DIR = os.path.join(os.getcwd(), "scripts")
os.makedirs(SCRIPTS_DIR, exist_ok=True)

path = os.path.join(SCRIPTS_DIR, "prepare_data.sh")
with open(path, 'w') as f:
    f.write('#!/bin/bash\n')
    f.write('set -e\n')
    f.write('source /data/hw3/speculators_venv/bin/activate\n')
    f.write('python /data/hw3/speculators/scripts/prepare_data.py \\\n')
    f.write('    --model /data/hw3/Qwen3-8B \\\n')
    f.write('    --data sharegpt \\\n')
    f.write('    --max-samples 3000 \\\n')
    f.write('    --seq-length 2048 \\\n')
    f.write('    --output /data/hw3/output\n')
os.chmod(path, os.stat(path).st_mode | stat.S_IEXEC)
print(f'Written: {path}')
print()
print('Run in terminal (speculators_venv):')
print(f'  bash {path}')
print()
print('Wait for completion before running Step 2.')
print()
# Optional: also write the dataset inspection script
insp_path = os.path.join(SCRIPTS_DIR, "inspect_sharegpt.py")
with open(insp_path, 'w') as f:
    f.write('from datasets import load_dataset\n')
    f.write('\n')
    f.write('ds = load_dataset("philschmid/sharegpt-raw", split="train")\n')
    f.write('print(f"Loaded {len(ds)} conversations")\n')
    f.write('for i in range(3):\n')
    f.write('    convs = ds[i]["conversations"]\n')
    f.write('    print(f"\\nSample {i}: {len(convs)} turns")\n')
    f.write('    for turn in convs[:2]:\n')
    f.write('        print(f"  [{turn[\"from\"]}: {turn[\"value\"][:120]}...")\n')
print(f'(Optional) inspect raw dataset: python {insp_path}')


### Step 2: Write & Launch Hidden State Generation (two terminals) — The Expensive Phase

This is the most time-consuming step. You run the verifier model (Qwen3-8B) on all training samples and save the intermediate hidden states to disk.

**Before running, open two monitoring terminals:**

```bash
# Monitoring Terminal A — disk usage (update every 5 s)
watch -n 5 df -h /data
```

```bash
# Monitoring Terminal B — GPU memory
watch -n 5 nvidia-smi
```

> **Experiment 2.2:** Before starting hidden state generation, measure baseline disk usage. Then every 30 minutes, record how much disk was consumed and estimate when you'll run out. This is your disk usage evidence log.

```
DISK USAGE LOG:
T=0:    519 GB free
T=30m:  ___ GB free (generated ≈ ___ samples)
T=60m:  ___ GB free (generated ≈ ___ samples)
...
Estimated GB per sample: ___
Max samples for available disk: ___
```

In [ ]:
# === STEP 2: Write server + generation scripts, run each in its own terminal ===
import os, stat

SCRIPTS_DIR = os.path.join(os.getcwd(), "scripts")
os.makedirs(SCRIPTS_DIR, exist_ok=True)

# ── Script A: vLLM server (Terminal 1, vllm_venv) ────────────────────
server_path = os.path.join(SCRIPTS_DIR, "launch_server.sh")
with open(server_path, 'w') as f:
    f.write('#!/bin/bash\n')
    f.write('set -e\n')
    f.write('source /data/hw3/vllm_venv/bin/activate\n')
    f.write('python /data/hw3/speculators/scripts/launch_vllm.py \\\n')
    f.write('    /data/hw3/Qwen3-8B \\\n')
    f.write('    -- \\\n')
    f.write('    --port 8000\n')
os.chmod(server_path, os.stat(server_path).st_mode | stat.S_IEXEC)
print(f'Written: {server_path}')

# ── Script B: hidden state generator (Terminal 2, speculators_venv) ──
gen_path = os.path.join(SCRIPTS_DIR, "generate_hidden_states.sh")
with open(gen_path, 'w') as f:
    f.write('#!/bin/bash\n')
    f.write('set -e\n')
    f.write('# speculators_venv: data_generation_offline.py imports from speculators\n')
    f.write('source /data/hw3/speculators_venv/bin/activate\n')
    f.write('python /data/hw3/speculators/scripts/data_generation_offline.py \\\n')
    f.write('    --model /data/hw3/Qwen3-8B \\\n')
    f.write('    --preprocessed-data /data/hw3/output \\\n')
    f.write('    --output /data/hw3/hidden_states \\\n')
    f.write('    --max-samples 3000\n')
os.chmod(gen_path, os.stat(gen_path).st_mode | stat.S_IEXEC)
print(f'Written: {gen_path}')
print()
print('Run in two separate tmux panes:')
print(f'  Terminal 1 (vllm_venv) — start first, wait for \'Application startup complete.\':')
print(f'    bash {server_path}')
print()
print(f'  Terminal 2 (speculators_venv) — once server is ready:')
print(f'    bash {gen_path}')
print()
print('Troubleshooting:')
print('  disk fills up  →  reduce --max-samples in generate_hidden_states.sh')


---
# Chapter 3 — Training the Draft Head
## Understanding What You're Actually Optimizing

## 3.1 The EAGLE-3 Loss Function

The draft head is trained to predict the verifier's next hidden state and (indirectly) the next token. The training loss has contributions from each **speculative position** (position 0, 1, 2, ...):

```
Position 0: Draft head predicts token t+1 given hidden states of t
Position 1: Draft head predicts token t+2 given its own prediction for t+1
Position 2: Draft head predicts token t+3 given its own prediction for t+2
```

Each position is harder than the previous — errors compound. This is why `val/loss` increases with position index.

---

## 3.2 Decoding the Training Metrics

After training, you'll see metrics like these. Before looking at the reference values, understand what each means:

| Metric | What It Measures |
|---|---|
| `val/loss_0_epoch` | Cross-entropy loss at position 0 (first draft token) |
| `val/full_acc_0_epoch` | P(draft token 0 = verifier's choice) |
| `val/cond_acc_0_epoch` | P(draft token 0 = verifier \| using teacher-forced input) |
| `val/loss_1_epoch` | Loss at position 1 (second draft token) |

**`full_acc` vs `cond_acc`:**
- `full_acc` (Full Accuracy): The draft head predicts position `k` using its *own* predictions as context for positions 0 to k-1. This is how it actually runs during inference — errors cascade.
- `cond_acc` (Conditional Accuracy): The draft head predicts position `k` using *ground truth* verifier tokens for positions 0 to k-1. This shows the upper bound — how good the head would be if earlier positions were always correct.

The gap between `full_acc` and `cond_acc` reveals **error propagation** — how much earlier position errors hurt later positions.

> **Experiment 3.1:** Record your position-wise metrics in a table. Then answer:
> 1. What is the `cond_acc` at position 2 minus `full_acc` at position 2? This is your error propagation gap.
> 2. If position 0 accuracy is very low (say, < 0.30), what would that tell you about the data generation step?


In [ ]:
# === EXPERIMENT 3.1: Training Metrics Evidence Log ===
# Fill in your actual training results after training completes.

# Reference values (from the homework)
reference_metrics = {
    'val/loss_0_epoch':     2.509,
    'val/full_acc_0_epoch': 0.463,
    'val/cond_acc_0_epoch': 0.463,
    'val/loss_1_epoch':     3.778,
    'val/full_acc_1_epoch': 0.181,
    'val/cond_acc_1_epoch': 0.364,
    'val/loss_2_epoch':     4.550,
    'val/full_acc_2_epoch': 0.069,
    'val/cond_acc_2_epoch': 0.320,
    'val/loss_epoch':       10.837,
    'epoch':                4,
}

# YOUR values (fill in after training)
your_metrics = {
    'val/loss_0_epoch':     None,  # fill in
    'val/full_acc_0_epoch': None,
    'val/cond_acc_0_epoch': None,
    'val/loss_1_epoch':     None,
    'val/full_acc_1_epoch': None,
    'val/cond_acc_1_epoch': None,
    'val/loss_2_epoch':     None,
    'val/full_acc_2_epoch': None,
    'val/cond_acc_2_epoch': None,
}

print("=== TRAINING METRICS COMPARISON ===")
print(f"{'Metric':<30} {'Reference':>12} {'Yours':>12} {'Diff':>8}")
print("-" * 65)
for k, ref_v in reference_metrics.items():
    if k == 'epoch': continue
    your_v = your_metrics.get(k)
    diff = f"{your_v - ref_v:+.3f}" if your_v is not None else "(pending)"
    your_str = f"{your_v:.3f}" if your_v is not None else "(pending)"
    print(f"{k:<30} {ref_v:>12.3f} {your_str:>12} {diff:>8}")

# Compute error propagation gap
print()
print("=== ERROR PROPAGATION ANALYSIS (Reference) ===")
for pos in range(3):
    full = reference_metrics.get(f'val/full_acc_{pos}_epoch', 0)
    cond = reference_metrics.get(f'val/cond_acc_{pos}_epoch', 0)
    gap = cond - full
    print(f"Position {pos}: full_acc={full:.3f}, cond_acc={cond:.3f}, gap={gap:.3f} "
          f"({'no propagation error' if gap < 0.001 else 'error propagates'})")

### Step 3: Write & Run the Training Script (after hidden states complete)

The script below writes `scripts/train_eagle3.sh` with:
- **TensorBoard logging** (`--logger tensorboard --log-dir /data/hw3/logs`) — live loss curves during training
- **Best-checkpoint tracking** (`--save-best`) — saves `checkpoints/checkpoint_best/` pointing to the epoch with the lowest `val/loss_epoch`, not necessarily the last epoch
- **stdout tee** — full log persisted to `/data/hw3/output/train_run.log` even if your terminal disconnects

**Watch during training (open a second terminal):**
```bash
# Option A: stream val metrics from the log file
tail -f /data/hw3/output/train_run.log | grep -E "val/"

# Option B: TensorBoard UI at http://localhost:6006
source /data/hw3/speculators_venv/bin/activate
tensorboard --logdir /data/hw3/logs --port 6006
```

**Key metrics to watch:**

| Metric | What to look for |
|---|---|
| `val/loss_0_epoch` | Should decrease each epoch; plateau = done training |
| `val/full_acc_0_epoch` | Target > 0.40; if stuck < 0.30 → bad hidden states |
| `val/loss_epoch` | Total loss across all positions; reference ≈ 10.84 |

> Best checkpoint is **not** always the last epoch — watch the val loss curve for the minimum.

In [2]:
# === STEP 3: Write the training script, then run it in a terminal ===
import os, stat

SCRIPTS_DIR = os.path.join(os.getcwd(), 'scripts')
os.makedirs(SCRIPTS_DIR, exist_ok=True)

train_path = os.path.join(SCRIPTS_DIR, 'train_eagle3.sh')
with open(train_path, 'w') as f:
    f.write('#!/bin/bash\n')
    f.write('set -e\n')
    f.write('source /data/hw3/speculators_venv/bin/activate\n')
    f.write('pip install -q tensorboard\n')
    f.write('python /data/hw3/speculators/scripts/train.py \\\n')
    f.write('    --verifier-name-or-path /data/hw3/Qwen3-8B \\\n')
    f.write('    --speculator-type eagle3 \\\n')
    f.write('    --data-path /data/hw3/output \\\n')
    f.write('    --hidden-states-path /data/hw3/hidden_states \\\n')
    f.write('    --save-path /data/hw3/output/checkpoints \\\n')
    f.write('    --epochs 5 \\\n')
    f.write('    --logger tensorboard \\\n')
    f.write('    --log-dir /data/hw3/logs \\\n')
    f.write('    --save-best \\\n')
    f.write('    2>&1 | tee /data/hw3/output/train_run.log\n')
os.chmod(train_path, os.stat(train_path).st_mode | stat.S_IEXEC)
print(f'Written: {train_path}')
print()
print('Run in terminal (inside a tmux session — training takes hours):')
print(f'  bash {train_path}')
print()
print('Monitor in a second terminal:')
print('  tail -f /data/hw3/output/train_run.log | grep -E "val/"')
print('  # or: tensorboard --logdir /data/hw3/logs --port 6006')


Written: /home/dishantghai/code/ai-perf-hw3-spec-decode-quant/scripts/train_eagle3.sh

Run in terminal (inside a tmux session — training takes hours):
  bash /home/dishantghai/code/ai-perf-hw3-spec-decode-quant/scripts/train_eagle3.sh

Watch during training:
  val/loss_0_epoch     — should decrease each epoch
  val/full_acc_0_epoch — target > 0.40
  Best checkpoint is NOT always the last epoch — check val loss curve


### Checkpoint Selection Strategy

> **Curiosity Probe:** The assignment mentions using the "best checkpoint" — but best by what metric? If position-0 accuracy improves but position-2 accuracy drops, which epoch's checkpoint should you use for serving?

Consider: In speculative decoding with `num_draft_tokens=2`, position 2 is never evaluated during serving. If you only use 2 draft tokens, only position-0 and position-1 accuracy matter for acceptance rate.

This is why **tuning `num_draft_tokens` and checkpoint selection are coupled decisions**.

**`--save-best` handles this automatically:** the trainer tracks `val/loss_epoch` (total loss across all positions) and writes a `checkpoint_best/` symlink whenever a new minimum is found. At the end of training, `checkpoint_best/` points to the optimal epoch — use that for serving instead of the last epoch.

```
CHECKPOINT SELECTION LOG:
Epoch 0: loss_0=___  full_acc_0=___  val/loss_epoch=___
Epoch 1: loss_0=___  full_acc_0=___  val/loss_epoch=___
Epoch 2: loss_0=___  full_acc_0=___  val/loss_epoch=___
Epoch 3: loss_0=___  full_acc_0=___  val/loss_epoch=___
Epoch 4: loss_0=___  full_acc_0=___  val/loss_epoch=___

Selected checkpoint: checkpoint_best/ → Epoch ___ (reason: lowest val/loss_epoch)
```

To confirm which epoch `checkpoint_best/` resolved to after training:
```bash
ls -la /data/hw3/output/checkpoints/checkpoint_best
```

---
# Chapter 4 — FP8 Quantization: Numbers on a Diet

## 4.1 Why FP8 and Not INT8 or INT4?

The H100 introduced hardware-native support for **FP8 tensor core operations** — a format that didn't exist in prior GPU generations. Let's understand why this matters.

### Floating Point Formats Side-by-Side

```
BF16:  [sign:1] [exponent:8] [mantissa:7]   → 16 bits, range ±3.4×10³⁸, 2-3 decimal digits
FP8:   [sign:1] [exponent:4] [mantissa:3]   → 8 bits,  range ±448,       ~1 decimal digit
INT8:  [sign:1] [magnitude:7]               → 8 bits,  range -128..127   (no exponent!)
```

FP8 keeps the **floating point structure** — it has an exponent. This means it can represent a wide dynamic range of values without needing careful per-layer calibration. INT8 must fit everything into [-128, 127] which requires either clipping (accuracy loss) or expensive calibration to find per-layer scale factors.

**Two FP8 flavors (IEEE 754 compatible):**
- `E4M3` (4 exponent, 3 mantissa bits): Better precision, smaller range. Good for weights.
- `E5M2` (5 exponent, 2 mantissa bits): Larger range, less precision. Good for gradients.

---

## 4.2 Dynamic vs Static Quantization

**Static quantization:**
- Run calibration data through the model, compute per-layer scale factors
- Scale factors are fixed at inference time
- Requires a representative calibration set
- Can fail if inference inputs have different distribution than calibration

**Dynamic quantization:**
- Compute the scale factor for each activation **on-the-fly** during inference
- No calibration data needed
- Slightly slower than static (scale computation overhead)
- More robust to out-of-distribution inputs

We use **dynamic FP8** for activations (weights can be statically quantized since they don't change at inference time).

> **Experiment 4.1:** Before quantizing, look at the weight distribution of a single layer. This builds intuition for why dynamic scaling is needed:


In [ ]:
# === EXPERIMENT 4.1: Weight Distribution Analysis ===
# This helps you understand WHY FP8 dynamic quantization is appropriate here.
# Run this BEFORE quantization to see what we're working with.

import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt
import numpy as np

# Load just the model config to inspect structure (don't load full weights for this cell)
from transformers import AutoConfig
config = AutoConfig.from_pretrained("Qwen/Qwen3-8B")

print("=== Qwen3-8B Architecture Summary ===")
print(f"Hidden dim:     {config.hidden_size}")
print(f"Num layers:     {config.num_hidden_layers}")
print(f"Attention heads: {config.num_attention_heads}")
print(f"KV heads:       {config.num_key_value_heads}  (GQA ratio: {config.num_attention_heads // config.num_key_value_heads}x)")
print(f"FFN dim:        {config.intermediate_size}")
print(f"Vocab size:     {config.vocab_size}")
print()

# Compute model weight categories for quantization
h = config.hidden_size
n_heads = config.num_attention_heads
n_kv = config.num_key_value_heads
ffn = config.intermediate_size
L = config.num_hidden_layers

head_dim = h // n_heads
q_proj = h * h        # Q projection
k_proj = h * (n_kv * head_dim)  # K projection
v_proj = h * (n_kv * head_dim)  # V projection  
o_proj = h * h        # Output projection
gate_proj = h * ffn   # SwiGLU gate
up_proj   = h * ffn   # SwiGLU up
down_proj = ffn * h   # SwiGLU down

attn_params = (q_proj + k_proj + v_proj + o_proj) * L
ffn_params  = (gate_proj + up_proj + down_proj) * L
embed_params = config.vocab_size * h  # lm_head + embedding (shared or not)

total = attn_params + ffn_params + embed_params
print(f"Attention params:  {attn_params/1e9:.2f}B  ({attn_params/total*100:.1f}%)")
print(f"FFN params:        {ffn_params/1e9:.2f}B  ({ffn_params/total*100:.1f}%)")
print(f"Embedding params:  {embed_params/1e9:.2f}B  ({embed_params/total*100:.1f}%)")
print(f"TOTAL:             {total/1e9:.2f}B")
print()
print("lm_head (vocab projection) is part of embedding params.")
print("If we exclude lm_head from FP8, we keep these params in BF16:")
print(f"  lm_head size: {config.vocab_size * h * 2 / 1e9:.2f} GB (BF16)")

## 4.3 Why Does lm_head Stay in BF16?

> **Curiosity Probe:** The homework explicitly says to keep `lm_head` unquantized. What makes it different from other linear layers?

Consider:
1. `lm_head` maps from hidden dimension (4096) → vocabulary size (152,064 for Qwen3)
2. The output of `lm_head` is the **logit vector** — a 152,064-dimensional vector where the values are used to compute token probabilities
3. The logit differences between similar tokens can be very small (e.g., the difference between predicting "cat" vs "cats" might be 0.1 in logit space)
4. FP8 E4M3 has only ~1 decimal digit of precision

**The risk:** If quantization noise in `lm_head` is larger than the logit gaps between similar tokens, the model will start making wrong choices. The vocabulary is large enough that this is a real concern, especially for:
- Rare tokens with naturally low logits
- Token disambiguation ("the" vs "The" vs "The.")
- Formatting tokens (newlines, spaces, punctuation)

Keeping `lm_head` in BF16 costs only `152,064 × 4096 × 1 byte ≈ 0.6 GB` extra — a small price for maintaining output quality.

---

## 4.4 Implementation — FP8 Dynamic Quantization

In [ ]:
# === FP8 Quantization Script ===
# Run this in comp_venv
# Follows: https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md

quantization_script = '''
# quantize_qwen3.py — run with: source comp_venv/bin/activate && python quantize_qwen3.py

from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.transformers import SparseAutoModelForCausalLM, oneshot
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3-8B"
OUTPUT_DIR = "./Qwen3-8B-FP8-Dynamic"

# Load the model in BF16 first
model = SparseAutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="bfloat16",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# FP8 dynamic quantization recipe:
# - weights: FP8 (quantized statically — they don't change at inference)
# - activations: FP8 dynamic (computed per-token at inference time)
# - lm_head: EXCLUDED (keeps BF16 precision for logit accuracy)
recipe = QuantizationModifier(
    targets="Linear",
    scheme="FP8_DYNAMIC",
    ignore=["lm_head"],   # <--- keep lm_head in BF16
)

# Apply quantization (one-shot: no calibration data needed for dynamic)
oneshot(model=model, recipe=recipe)

# Verify quantization was applied
import json
quant_config = model.config.to_dict().get("quantization_config", {})
print("Quantization config:", json.dumps(quant_config, indent=2))

# Save — do NOT overwrite the original BF16 checkpoint
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nSaved quantized model to: {OUTPUT_DIR}")
'''

print("Write this to quantize_qwen3.py and run in comp_venv")
print()
print(quantization_script)

### Step 5: Verify the Quantized Model

In [ ]:
# Verification: Inspect the saved model's config before benchmarking
# Run in comp_venv or any env with transformers installed

verification = """
python -c "
import json, os
from pathlib import Path

config_path = Path('./Qwen3-8B-FP8-Dynamic/config.json')
config = json.loads(config_path.read_text())

quant = config.get('quantization_config', {})
print('=== Quantization Config ===')
print(f\"Method: {quant.get('quant_type', 'NOT FOUND')}\")

# Check a few layers for FP8 annotation
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained('./Qwen3-8B-FP8-Dynamic')
print(f\"Model type: {cfg.model_type}\")
print(f\"Quantization: {getattr(cfg, 'quantization_config', 'NONE')}\")
"
"""

print("Run this to verify quantization before serving:")
print(verification)

print()
print("=== Expected Quantization Properties ===")
expected = {
    "Quantization method": "compressed tensors",
    "Weight format": "FP8",
    "Activation format": "dynamic FP8",
    "Target modules": "linear layers",
    "Ignored module": "lm_head",
}
for k, v in expected.items():
    print(f"  {k}: {v}")

---
# Chapter 5 — The Grand Benchmark
## Setting Up a Fair Experiment

## 5.1 Experimental Design Principles

Before running any benchmark, commit to the experimental protocol. Any deviation invalidates the comparison.

**Fixed across all four configurations:**
- Same benchmark dataset (`philschmid/mt-bench`)
- Same number of prompts (`--num-prompts 80`)
- Same concurrency level (`--max-concurrency 8`)
- Prefix caching disabled (unless studying it intentionally)
- Fixed seed where vLLM supports it

**Variable (to be tuned per configuration):**
- Number of speculative draft tokens (only for spec decoding configs)

---

## 5.2 Configuration Map

| Config | Model | Draft Head | Expected Gain Source |
|---|---|---|---|
| C1: Baseline | Qwen3-8B (BF16) | None | — |
| C2: Spec Dec | Qwen3-8B (BF16) | EAGLE-3 head | More tokens per verifier pass |
| C3: FP8 Quant | Qwen3-8B (FP8) | None | Faster per-token memory reads |
| C4: Combined | Qwen3-8B (FP8) | EAGLE-3 head | Both effects compound |

> **Prediction Exercise:** Before running, fill in your predicted output tok/s for each config:
> ```
> C1 (Baseline):  ___ tok/s  (run first, use as anchor)
> C2 (Spec Dec):  ___ tok/s  (predict % improvement over C1)
> C3 (FP8):       ___ tok/s  (predict % improvement over C1)
> C4 (Combined):  ___ tok/s  (is it additive? multiplicative? explain)
> ```

---

## 5.3 The Draft Token Tuning Problem

For C2 and C4, you must tune `num_speculative_tokens`. This is not a one-size-fits-all value.

**Why C4 uses fewer draft tokens than C2 (hint from reference results):**
- C2 uses 2 draft tokens, achieves acceptance length 1.45
- C4 uses 1 draft token, achieves acceptance length 1.36

But wait — why is less better for C4? Let's find out through experiments.

> **Experiment 5.1: Draft Token Sweep** — Run C2 (spec decoding, BF16) with 1, 2, 3, and 4 draft tokens. Record the results:

In [ ]:
**▶ Commands to run the draft token sweep** — `vllm_venv`, Terminal 1 + 2 pattern.  
Restart Terminal 1 for each value of `DRAFT_TOKENS` (1, 2, 3, 4). Keep Terminal 2 command identical.

```bash
# ── Terminal 1: C2 sweep — BF16 + EAGLE-3, vary draft tokens ──
source /data/hw3/vllm_venv/bin/activate
vllm serve /data/hw3/Qwen3-8B \
    --speculative-config '{"method":"eagle3","model":"/data/hw3/output/checkpoints/best_checkpoint","num_speculative_tokens":DRAFT_TOKENS}' \
    --no-enable-prefix-caching
```

```bash
# ── Terminal 2: benchmark (same every run) ──
source /data/hw3/vllm_venv/bin/activate
vllm bench serve \
    --model /data/hw3/Qwen3-8B \
    --dataset-name hf \
    --dataset-path philschmid/mt-bench \
    --max-concurrency 8 \
    --num-prompts 80
```

For the **C4 sweep** (FP8 + EAGLE-3), swap the model path in Terminal 1:

```bash
# ── Terminal 1: C4 sweep — FP8 + EAGLE-3, vary draft tokens ──
source /data/hw3/vllm_venv/bin/activate
vllm serve /data/hw3/Qwen3-8B-FP8-Dynamic \
    --speculative-config '{"method":"eagle3","model":"/data/hw3/output/checkpoints/best_checkpoint","num_speculative_tokens":DRAFT_TOKENS}' \
    --no-enable-prefix-caching
```

Fill the tables below with results from each run, then proceed to the final benchmark.

---

# === EXPERIMENT 5.1: Draft Token Sweep Evidence Log ===
# Fill this in as you run each configuration.

import pandas as pd

# Config C2: BF16 + EAGLE-3, varying draft tokens
c2_sweep = {
    'num_draft_tokens': [1, 2, 3, 4],
    'output_toks_per_s':  [None, None, None, None],  # fill in
    'acceptance_rate_%':  [None, None, None, None],  # fill in
    'acceptance_length':  [None, None, None, None],  # fill in
    'mean_tpot_ms':       [None, None, None, None],  # fill in
}

# Config C4: FP8 + EAGLE-3, varying draft tokens
c4_sweep = {
    'num_draft_tokens': [1, 2, 3, 4],
    'output_toks_per_s':  [None, None, None, None],  # fill in
    'acceptance_rate_%':  [None, None, None, None],  # fill in
    'acceptance_length':  [None, None, None, None],  # fill in
    'mean_tpot_ms':       [None, None, None, None],  # fill in
}

print("=== C2: BF16 + EAGLE-3 Draft Token Sweep ===")
df_c2 = pd.DataFrame(c2_sweep)
print(df_c2.to_string(index=False))

print()
print("=== C4: FP8 + EAGLE-3 Draft Token Sweep ===")
df_c4 = pd.DataFrame(c4_sweep)
print(df_c4.to_string(index=False))

print()
print("ANALYSIS QUESTIONS:")
print("1. For C2: at what num_draft_tokens does throughput peak?")
print("2. For C4: is the peak at the same num_draft_tokens as C2?")
print("3. When adding more draft tokens reduces throughput, what is happening?")
print("   (Hint: look at acceptance_length vs num_draft_tokens ratio)")

### 5.4 Final Benchmark Commands (All Four Configurations)

> Run these **after** the sweep above has identified the optimal `DRAFT_TOKENS` for C2 and C4.  
> Pattern: start server in Terminal 1, benchmark in Terminal 2, Ctrl-C Terminal 1 between configs.

**Environment:** `vllm_venv` | **Working dir:** `/data/hw3`


**Environment:** `vllm_venv` | **Working dir:** `/data/hw3`

Before running: set your paths and chosen draft token count.

```
DRAFT_HEAD=/data/hw3/output/checkpoints/best_checkpoint
FP8_MODEL=/data/hw3/Qwen3-8B-FP8-Dynamic
BASE_MODEL=/data/hw3/Qwen3-8B
BENCH="--dataset-name hf --dataset-path philschmid/mt-bench --max-concurrency 8 --num-prompts 80"
DRAFT_TOKENS=2   # replace with your tuned value for C2
```

---

### C1 — Baseline (BF16, no spec decoding)

```bash
# Terminal 1
vllm serve /data/hw3/Qwen3-8B --no-enable-prefix-caching

# Terminal 2
vllm bench serve --model /data/hw3/Qwen3-8B \
    --dataset-name hf --dataset-path philschmid/mt-bench \
    --max-concurrency 8 --num-prompts 80
```

---

### C2 — Spec Decoding (BF16 + EAGLE-3)

```bash
# Terminal 1  (replace 2 with your tuned DRAFT_TOKENS)
vllm serve /data/hw3/Qwen3-8B \
    --speculative-config '{"method":"eagle3","model":"/data/hw3/output/checkpoints/best_checkpoint","num_speculative_tokens":2}' \
    --no-enable-prefix-caching

# Terminal 2
vllm bench serve --model /data/hw3/Qwen3-8B \
    --dataset-name hf --dataset-path philschmid/mt-bench \
    --max-concurrency 8 --num-prompts 80
```

---

### C3 — FP8 Quantized (no spec decoding)

```bash
# Terminal 1
vllm serve /data/hw3/Qwen3-8B-FP8-Dynamic --no-enable-prefix-caching

# Terminal 2
vllm bench serve --model /data/hw3/Qwen3-8B-FP8-Dynamic \
    --dataset-name hf --dataset-path philschmid/mt-bench \
    --max-concurrency 8 --num-prompts 80
```

---

### C4 — Combined (FP8 + EAGLE-3)

```bash
# Terminal 1  (replace 1 with your tuned DRAFT_TOKENS for C4)
vllm serve /data/hw3/Qwen3-8B-FP8-Dynamic \
    --speculative-config '{"method":"eagle3","model":"/data/hw3/output/checkpoints/best_checkpoint","num_speculative_tokens":1}' \
    --no-enable-prefix-caching

# Terminal 2
vllm bench serve --model /data/hw3/Qwen3-8B-FP8-Dynamic \
    --dataset-name hf --dataset-path philschmid/mt-bench \
    --max-concurrency 8 --num-prompts 80
```

> Ctrl-C Terminal 1 between each config. Wait for "Application startup complete." before starting Terminal 2.


### Results Evidence Log — Fill In As You Run

In [ ]:
# === GRAND BENCHMARK RESULTS TABLE ===
import pandas as pd

# Reference results (from homework)
reference = [
    {"config": "Baseline",            "duration_s": 24.35, "req_per_s": 3.29, "out_toks_per_s": 841.22,  "mean_ttft_ms": 576.17, "mean_tpot_ms": 7.28,  "acceptance_%": None,   "draft_tokens": None},
    {"config": "Spec Decoding",       "duration_s": 16.27, "req_per_s": 4.92, "out_toks_per_s": 1258.65, "mean_ttft_ms": 78.17,  "mean_tpot_ms": 5.76,  "acceptance_%": 22.48,  "draft_tokens": 2},
    {"config": "FP8 Quant",           "duration_s": 13.06, "req_per_s": 6.12, "out_toks_per_s": 1566.56, "mean_ttft_ms": 51.18,  "mean_tpot_ms": 4.90,  "acceptance_%": None,   "draft_tokens": None},
    {"config": "FP8 + Spec Decoding", "duration_s": 11.59, "req_per_s": 6.90, "out_toks_per_s": 1766.55, "mean_ttft_ms": 30.24,  "mean_tpot_ms": 4.28,  "acceptance_%": 36.50,  "draft_tokens": 1},
]

# YOUR results (fill in)
your_results = [
    {"config": "Baseline",            "out_toks_per_s": None, "mean_ttft_ms": None, "mean_tpot_ms": None, "acceptance_%": None, "draft_tokens": None},
    {"config": "Spec Decoding",       "out_toks_per_s": None, "mean_ttft_ms": None, "mean_tpot_ms": None, "acceptance_%": None, "draft_tokens": None},
    {"config": "FP8 Quant",           "out_toks_per_s": None, "mean_ttft_ms": None, "mean_tpot_ms": None, "acceptance_%": None, "draft_tokens": None},
    {"config": "FP8 + Spec Decoding", "out_toks_per_s": None, "mean_ttft_ms": None, "mean_tpot_ms": None, "acceptance_%": None, "draft_tokens": None},
]

print("=== REFERENCE RESULTS ===")
ref_df = pd.DataFrame(reference)[['config', 'out_toks_per_s', 'mean_ttft_ms', 'mean_tpot_ms', 'acceptance_%', 'draft_tokens']]
print(ref_df.to_string(index=False))

print()
print("=== YOUR RESULTS ===")
your_df = pd.DataFrame(your_results)
print(your_df.to_string(index=False))

# Compute speedups
print()
print("=== REFERENCE SPEEDUPS vs BASELINE ===")
baseline_toks = 841.22
for r in reference:
    if r['out_toks_per_s'] and r['config'] != 'Baseline':
        speedup = r['out_toks_per_s'] / baseline_toks
        print(f"  {r['config']}: {speedup:.2f}× ({(speedup-1)*100:.1f}% improvement)")

# Scoring check
print()
print("=== SCORING THRESHOLDS ===")
thresholds = [
    ("Spec decoding > 1250 tok/s", 1250, 25, "Spec Decoding"),
    ("FP8 quant > 1550 tok/s",     1550, 10, "FP8 Quant"),
    ("Combined > 1750 tok/s",       1750, 15, "FP8 + Spec Decoding"),
]
total_possible = sum(pts for _, _, pts, _ in thresholds)
for desc, threshold, pts, config_name in thresholds:
    ref_val = next(r['out_toks_per_s'] for r in reference if r['config'] == config_name)
    status = "PASS" if ref_val > threshold else "FAIL"
    print(f"  {desc}: {ref_val:.1f} → {status} ({pts} pts)")

---
## 5.4 Interpreting the Results

Once you have all four rows filled in, work through these questions. Do NOT read the answers first — generate your own explanation from the numbers, then check.

---

### Question A: Why Does Spec Decoding Improve Throughput Even at 22% Acceptance Rate?

First intuition: 22% sounds low. If only 22% of drafts are accepted, is it even worth doing?

But look at the metric: **acceptance length = 1.45**, not 0.22.

Acceptance rate is per-draft-token (what % of individual draft tokens are accepted), while acceptance length is the mean number of tokens accepted per draft-and-verify cycle.

> **Experiment 5.2:** Use the reference speculative decoding details to work this out:

```
Reference: 14,088 draft cycles, 28,176 draft tokens, 6,334 accepted tokens

Mean tokens per verify cycle (accepted + corrected) = ?
Without spec decoding: 14,088 verify cycles, each producing 1 token
With spec decoding:    14,088 verify cycles, each producing ~1.45 tokens

The verifier does the same number of forward passes — but each pass produces more tokens.
```


In [ ]:
# === EXPERIMENT 5.2: Acceptance Rate Analysis ===

# Reference spec decoding numbers
c2_drafts = 14088
c2_draft_tokens = 28176
c2_accepted_tokens = 6334
c2_num_draft_per_cycle = 2

# Reference FP8 + spec decoding
c4_drafts = 14954
c4_draft_tokens = 14954
c4_accepted_tokens = 5458
c4_num_draft_per_cycle = 1

print("=== C2: Spec Decoding (BF16) ===")
per_token_accept = c2_accepted_tokens / c2_draft_tokens
tokens_per_cycle = (c2_accepted_tokens + c2_drafts) / c2_drafts  # +1 for corrected token per cycle
print(f"Draft cycles:              {c2_drafts}")
print(f"Draft tokens per cycle:    {c2_draft_tokens / c2_drafts:.2f}")
print(f"Per-token acceptance rate: {per_token_accept*100:.1f}%")
print(f"Tokens produced per cycle: {tokens_per_cycle:.2f}  (accepted + 1 corrected)")
print(f"Speedup vs no spec (1.0 tok/cycle): {tokens_per_cycle:.2f}×")

print()
print("=== C4: FP8 + Spec Decoding ===")
per_token_accept_c4 = c4_accepted_tokens / c4_draft_tokens
tokens_per_cycle_c4 = (c4_accepted_tokens + c4_drafts) / c4_drafts
print(f"Draft cycles:              {c4_drafts}")
print(f"Draft tokens per cycle:    {c4_draft_tokens / c4_drafts:.2f}")
print(f"Per-token acceptance rate: {per_token_accept_c4*100:.1f}%")
print(f"Tokens produced per cycle: {tokens_per_cycle_c4:.2f}  (accepted + 1 corrected)")
print(f"Speedup vs no spec (1.0 tok/cycle): {tokens_per_cycle_c4:.2f}×")

print()
print("KEY INSIGHT: Even with <50% acceptance rate, spec decoding wins because:")
print("  - The verifier forward pass cost is dominated by MEMORY BANDWIDTH (loading weights)")
print("  - Verifying K tokens in parallel costs ~K× FLOPs but same memory bandwidth")
print("  - So verified tokens are nearly 'free' in time, just slightly more compute")

print()
print("Now compare: why does C4 use 1 draft token but C2 uses 2?")
print(f"  C2: draft overhead = {c2_draft_tokens} tokens to produce {c2_accepted_tokens} extra accepted tokens")
print(f"  C4: draft overhead = {c4_draft_tokens} tokens to produce {c4_accepted_tokens} extra accepted tokens")
print("  The FP8 verifier is already faster — fewer wasted drafts needed to win")

---
# Chapter 6 — The Interaction Problem
## Does Quantization Break Speculative Decoding?

This is the subtlest part of the whole assignment.

> **Curiosity Probe:** The draft head was trained on hidden states from the **BF16** verifier. When you quantize the verifier to FP8, its hidden states change slightly. Does the draft head still work? How much does acceptance rate change?

From the reference results:
- C2 (BF16 + spec): acceptance rate = **22.48%**, acceptance length = **1.45**
- C4 (FP8 + spec): acceptance rate = **36.50%**, acceptance length = **1.36**

Wait — acceptance rate is *higher* with FP8? And yet acceptance length is *lower*? How can both be true?

This requires careful analysis:

In [ ]:
# === EXPERIMENT 6.1: Analyzing the Acceptance Rate Paradox ===

# C2: 2 draft tokens, acceptance rate 22.48%, acceptance length 1.45
# C4: 1 draft token, acceptance rate 36.50%, acceptance length 1.36

print("=== The Acceptance Rate Paradox ===")
print()
print("C2 (BF16 + 2 draft tokens):")
print("  acceptance rate per token = 22.48%")
print("  acceptance length = 1.45")
print("  If token 0 is accepted at 22.48%, that's the bottleneck.")
print("  Expected length from 2 drafts: sum over geometric distribution")

# Model: each draft token accepted with probability p, independently
# (simplified — reality is conditional)
p_c2 = 0.2248
p_c4 = 0.3650

# Expected accepted tokens from 2 drafts
# E[accepted | 2 drafts] ≈ p0 + p0*p1 (where p1 = conditional acceptance)
# We observe acceptance length 1.45 with 2 drafts
# For 1 draft: E[accepted | 1 draft] ≈ p0

print()
print("C4 (FP8 + 1 draft token):")
print("  acceptance rate per token = 36.50%")
print("  acceptance length = 1.36")
print("  Higher per-token rate but fewer total accepted because only 1 draft is attempted")

print()
print("RESOLUTION:")
print("  'acceptance rate' = per-token acceptance rate")
print("  'acceptance length' = mean tokens accepted per draft cycle")
print("  These are DIFFERENT metrics! With 1 draft token, acceptance length ≤ 1.")
print("  With 2 draft tokens, acceptance length can reach up to 2.")
print()
print("FP8 quant may change the output distribution slightly:")
print("  → More 'peaky' logits (FP8 rounding reinforces the top token)")
print("  → Easier for draft head to predict the top token → higher rate")
print("  → But only 1 draft token is used, capping acceptance length at 1.")

print()
print("KEY QUESTION: How did the EAGLE-3 draft head, trained on BF16 hidden states,")
print("achieve HIGHER acceptance rate with the FP8 verifier?")
print()
print("Hypotheses to test:")
print("  H1: FP8 rounding makes verifier outputs more predictable (sharper distributions)")
print("  H2: The acceptance mechanism in speculative decoding is robust to small distribution shifts")
print("  H3: FP8 weights approximate BF16 well enough that hidden states differ minimally")
print()
print("Which hypothesis do you think explains the result? Run a quick check:")
print("  - Compare output probabilities (top-k) between BF16 and FP8 models on the same prompt")

### Bonus Experiment: BF16 vs FP8 Output Distribution

This experiment is optional but deeply informative.

In [ ]:
# === BONUS EXPERIMENT: Compare BF16 vs FP8 output logits ===
# Run this in comp_venv with both models loaded

# This shows how much quantization changes the output distribution
# Key metric: KL divergence between BF16 and FP8 output distributions

logit_comparison_script = '''
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

# Test prompts — deliberately chosen to be tricky
test_prompts = [
    "The capital of France is",
    "def fibonacci(n):",
    "In 2024, the most popular programming language was",
]

results = []
for model_path, label in [("Qwen/Qwen3-8B", "BF16"), ("./Qwen3-8B-FP8-Dynamic", "FP8")]:
    model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", torch_dtype=torch.bfloat16)
    model.eval()
    model_results = []
    with torch.no_grad():
        for prompt in test_prompts:
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            logits = model(**inputs).logits[0, -1]  # last token
            probs = F.softmax(logits, dim=-1)
            top5_probs, top5_ids = probs.topk(5)
            top5_tokens = [tokenizer.decode([t]) for t in top5_ids]
            model_results.append(list(zip(top5_tokens, top5_probs.tolist())))
    results.append((label, model_results))
    del model
    torch.cuda.empty_cache()

# Compare
for i, prompt in enumerate(test_prompts):
    print(f"\nPrompt: {prompt!r}")
    for label, res in results:
        print(f"  {label}: {[(t.strip(), f'{p:.3f}') for t, p in res[i][:3]]}")
'''

print("Bonus experiment — run in comp_venv:")
print(logit_comparison_script)

---
# Chapter 7 — The Central Mystery
## Which Should Be Done First: Quantization or Spec Decoding Training?

This is the question the homework asked in the opening. Before reading further, write down your current hypothesis and the evidence you have so far.

```
MY HYPOTHESIS (based on experiments so far):
Order: ___________
Reason: ___________
Evidence supporting this: ___________
```

---

## 7.1 The Train-on-What-You-Serve Principle

The EAGLE-3 draft head learns to predict the verifier model's behavior. Specifically, it learns:

> Given the verifier's hidden states at position `t`, predict the verifier's hidden states at position `t+1`.

The **hidden states are model-specific**. A BF16 verifier and an FP8 verifier have different internal representations for the same input — not wildly different, but measurably different.

If you train the draft head on BF16 hidden states, then quantize the verifier to FP8:
```
Training distribution: P_BF16(h_{t+1} | h_t)
Serving distribution:  P_FP8(h_{t+1} | h_t)

Distribution shift = ||P_BF16 - P_FP8||
```

The draft head must generalize across this shift. If the shift is small, it works fine. If large, acceptance rate drops.

If instead you **quantize first, then train the draft head on FP8 hidden states**:
```
Training distribution: P_FP8(h_{t+1} | h_t)
Serving distribution:  P_FP8(h_{t+1} | h_t)

Distribution shift = 0
```

The draft head is trained on exactly the verifier it will serve with. No distribution shift. Maximum acceptance rate.

---

## 7.2 The Hidden State Generation Bottleneck

Here's a practical consideration that reinforces the theory:

**Hidden state generation** in offline EAGLE-3 training runs the verifier model to produce training data. If you run the BF16 verifier for hidden state generation, you get BF16 hidden states. If you run the FP8 verifier, you get FP8 hidden states.

Generating FP8 hidden states is **faster** (FP8 verifier is faster) and **directly matches the serving environment**.

So the optimal workflow is:

```
1. Start with base model: Qwen3-8B (BF16)
2. Quantize to FP8:       Qwen3-8B → Qwen3-8B-FP8-Dynamic
3. Generate hidden states: run Qwen3-8B-FP8-Dynamic on ShareGPT
4. Train draft head:       train EAGLE-3 on FP8 hidden states
5. Serve:                  Qwen3-8B-FP8-Dynamic + EAGLE-3 draft head
```

Every stage operates on consistent representations. The draft head learns to predict exactly what the production verifier will produce.

---

> **Experiment 7.1 (Critical):** To empirically verify this matters, try the opposite order:
> 1. Train the draft head on BF16 hidden states (current C2 setup)
> 2. Use the same draft head with the FP8 verifier (C4 setup)
> 3. Compare C4's acceptance rate to what you'd expect if the draft head were trained on FP8 states
> 
> The gap between the two is your **empirical distribution shift cost**.

```
EXPERIMENT 7.1 RESULTS:
C4 (draft trained on BF16, serving FP8): acceptance rate = ____%
Projected C4* (draft trained on FP8, serving FP8): est. acceptance rate = ____%
Distribution shift cost: ___ percentage points in acceptance rate
```


In [ ]:
# === EXPERIMENT 7.2: Order-of-Operations Analysis ===
# Build the evidence for the final answer.

print("=== ORDER OF OPERATIONS: THEORETICAL ANALYSIS ===")
print()
print("Option A: Train draft head first (on BF16), then quantize verifier")
print("  Step 1: Qwen3-8B (BF16) → generate hidden states")
print("  Step 2: Train EAGLE-3 draft head on BF16 hidden states")
print("  Step 3: Quantize Qwen3-8B → Qwen3-8B-FP8-Dynamic")
print("  Step 4: Serve FP8 model + BF16-trained draft head")
print("  RISK: Distribution shift between training (BF16) and serving (FP8) environments")
print()
print("Option B: Quantize first, then train draft head on quantized model")
print("  Step 1: Quantize Qwen3-8B → Qwen3-8B-FP8-Dynamic")
print("  Step 2: Generate hidden states from FP8 model")
print("  Step 3: Train EAGLE-3 draft head on FP8 hidden states")
print("  Step 4: Serve FP8 model + FP8-trained draft head")
print("  BENEFIT: Zero distribution shift. Draft head matches serving environment exactly.")
print()

print("=== PRACTICAL CONSIDERATIONS ===")
print()
considerations = [
    ("Hidden state disk usage",      "Same for both options", "Same for both options"),
    ("Hidden state gen speed",       "BF16 model (slower)",   "FP8 model (faster, ~1.5-2x)"),
    ("Acceptance rate potential",    "Lower (distribution shift)", "Higher (exact match)"),
    ("Re-training needed if quant changes", "YES — must retrain", "YES — must retrain (same)"),
    ("Flexibility for future quant formats", "Draft works with BF16 baseline", "Draft coupled to FP8"),
]

print(f"{'Factor':<35} {'Option A (BF16 first)':<30} {'Option B (FP8 first)':<30}")
print("-" * 95)
for factor, a, b in considerations:
    print(f"{factor:<35} {a:<30} {b:<30}")

print()
print("VERDICT: Option B is the correct approach for production deployment.")
print("  The draft head should always be trained on the same model it will serve with.")
print("  Quantization defines the 'true' verifier. Train the draft head on that verifier.")

---
# Chapter 8 — Evidence Synthesis
## Writing Your Final Technical Argument

You now have:
1. Baseline benchmark (C1) — the reference point
2. Spec decoding benchmark (C2) — quantified gain from EAGLE-3
3. FP8 benchmark (C3) — quantified gain from quantization
4. Combined benchmark (C4) — quantified gain from both
5. Acceptance rate analysis — why and how spec decoding helps
6. Distribution shift analysis — why order matters

Now synthesize all of this into a technical argument. Answer the homework's central question using your own evidence.

---

## 8.1 Final Evidence Table

In [ ]:
# === FINAL EVIDENCE TABLE ===
# Fill in all values from YOUR experiments.

print("=" * 80)
print("FINAL BENCHMARK RESULTS")
print("=" * 80)

# Your results go here
your_final_results = {
    "Baseline":            {"out_toks_s": None,  "ttft_ms": None, "tpot_ms": None, "accept_%": "N/A", "speedup": 1.0},
    "Spec Decoding":       {"out_toks_s": None,  "ttft_ms": None, "tpot_ms": None, "accept_%": None,  "speedup": None},
    "FP8 Quant":           {"out_toks_s": None,  "ttft_ms": None, "tpot_ms": None, "accept_%": "N/A", "speedup": None},
    "FP8 + Spec Decoding": {"out_toks_s": None,  "ttft_ms": None, "tpot_ms": None, "accept_%": None,  "speedup": None},
}

# When filled in, compute speedups
baseline_toks = your_final_results["Baseline"]["out_toks_s"]
if baseline_toks:
    for config, r in your_final_results.items():
        if r["out_toks_s"] and config != "Baseline":
            r["speedup"] = round(r["out_toks_s"] / baseline_toks, 2)

print(f"{'Config':<25} {'tok/s':<10} {'TTFT ms':<10} {'TPOT ms':<10} {'Accept%':<10} {'Speedup':<8}")
print("-" * 75)
for config, r in your_final_results.items():
    print(f"{config:<25} {str(r['out_toks_s']):<10} {str(r['ttft_ms']):<10} {str(r['tpot_ms']):<10} {str(r['accept_%']):<10} {str(r['speedup']):<8}")

print()
print("SCORING CHECK:")
thresholds = [
    ("Spec decoding", 1250, 25),
    ("FP8 Quant",     1550, 10),
    ("Combined",      1750, 15),
]
score = 0
configs_map = {"Spec decoding": "Spec Decoding", "FP8 Quant": "FP8 Quant", "Combined": "FP8 + Spec Decoding"}
for desc, threshold, pts in thresholds:
    val = your_final_results.get(configs_map[desc], {}).get("out_toks_s")
    if val is not None:
        passed = val > threshold
        score += pts if passed else 0
        print(f"  {desc}: {val:.1f} {'≥' if passed else '<'} {threshold} → {'PASS (+' + str(pts) + 'pts)' if passed else 'FAIL (0 pts)'}")
    else:
        print(f"  {desc}: (not yet measured)")
print(f"  TOTAL SCORE: {score}/50")

---
## 8.2 The Answer — Write It In Your Own Words

Use this template to write your final technical explanation. Fill each section with your own evidence and reasoning.

```
=== FINAL TECHNICAL REPORT ===

QUESTION: Which optimization should be applied first — quantization or speculative decoding training?

ANSWER: ___________

EVIDENCE:

1. Baseline Performance
   - Output throughput: ___ tok/s
   - TPOT: ___ ms (indicates ___-bound regime)

2. FP8 Quantization Effect (C3 vs C1)
   - Throughput change: ___ → ___ tok/s (+___%)  
   - TPOT change: ___ → ___ ms
   - Mechanism: FP8 halves weight load bandwidth from ___ GB to ___ GB per token

3. Speculative Decoding Effect (C2 vs C1)
   - Throughput change: ___ → ___ tok/s (+___%)  
   - Acceptance rate: ___%, acceptance length: ___
   - Mechanism: Each verifier pass produces ~___ tokens instead of 1

4. Combined Effect (C4 vs C1)
   - Throughput change: ___ → ___ tok/s (+___% vs baseline, +___% vs best single)
   - The gains are roughly ___-additive (multiplicative/subadditive/superadditive)

5. Order-of-Operations Argument
   - The EAGLE-3 draft head learns P(h_{t+1} | h_t) for a specific verifier model
   - If quantization changes the verifier's hidden state distribution, a BF16-trained draft head
     incurs distribution shift at serving time, reducing acceptance rate
   - Correct order: (1) Quantize verifier → (2) Generate hidden states from FP8 verifier →
     (3) Train draft head on FP8 states → (4) Serve FP8 verifier + FP8-trained draft head
   - Empirical support: C4 achieved acceptance rate of ___% with the BF16-trained draft head;
     a FP8-trained draft head would theoretically achieve higher acceptance rate
     (less distribution shift between train and serve environments)

CONCLUSION:
Quantization should be applied FIRST for two reasons:
  (a) TECHNICAL: The draft head must be trained to match the verifier it will actually serve with.
      Training on BF16 then serving with FP8 introduces unnecessary distribution shift.
  (b) PRACTICAL: Hidden state generation is faster when using the FP8 verifier, reducing
      the offline training pipeline cost.

The correct pipeline for maximum performance:
  BF16 model → FP8 quantization → hidden state generation → EAGLE-3 training → serving
```


---
# Epilogue — Where This Journey Ends (And What Comes Next)

## What You've Learned

| Concept | What You Now Know |
|---|---|
| Roofline model | LLM decoding is memory-bandwidth-bound, not compute-bound |
| Speculative decoding | Uses parallel verification to get >1 token per verifier pass |
| EAGLE-3 | Trains a draft head on hidden states; offline needs one GPU but 100+ GB disk |
| Full vs conditional accuracy | Measures error cascade vs upper bound for draft head quality |
| FP8 dynamic quantization | Halves weight bandwidth, works natively on H100 |
| lm_head exclusion | Preserves logit precision for vocabulary-level decisions |
| Distribution shift | Draft heads must be trained on the verifier they'll serve with |
| Optimal ordering | Quantize first → generate FP8 hidden states → train draft head |

---

## What You Could Explore Next

1. **INT4 / AWQ Quantization:** Even more aggressive than FP8. Does the draft head still generalize? What's the acceptance rate hit?

2. **Tree-based Speculative Decoding:** Instead of a single draft sequence, propose a tree of possibilities. The verifier evaluates all branches in one pass.

3. **Online EAGLE-3 Training:** What happens to acceptance rate if you train on 10× more samples using two GPUs? Is there a diminishing return?

4. **Task-specific Draft Heads:** Train a draft head on code-only data. Benchmark on coding tasks vs general tasks. Quantify the domain benefit.

5. **KV Cache Quantization:** The KV cache is also memory-intensive. FP8 KV cache is supported by H100 and vLLM. Does combining KV8 + FP8 weights + EAGLE-3 give further gains?

6. **Chunked Prefill vs Continuous Batching:** How does TTFT change across these configurations? Is speculative decoding better or worse for latency-sensitive applications?

---

## Curiosity Probes — Remaining Open Questions

Leave these as journal prompts for yourself:

1. At what batch size does speculative decoding start to hurt rather than help? (The verifier is no longer bandwidth-bound at high batch.)

2. If you doubled the number of training samples from 3,000 to 6,000, by how much would acceptance rate improve? Is there a saturation point?

3. The EAGLE-3 draft head itself takes up GPU memory. At what model size does the draft head become a significant fraction of the verifier? Does this change the calculus?

4. The homework uses `mt-bench` for benchmarking, but the draft head is trained on ShareGPT. Are these distributions aligned? Would training on `mt-bench`-style data improve acceptance rate on this benchmark?

5. FP8 quantization changes the hidden states slightly. Could you fine-tune the BF16 draft head on a small amount of FP8 hidden states to "bridge" the distribution shift rather than retraining from scratch?

---

*This guide is a map, not a recipe. The territory is your GPU, your experiments, and the numbers only you will generate.*


---
# Chapter 9 — Filling the Submission Notebook
## Exactly What Goes Where, and How to Package It

The homework notebook (`spec_dec+quantization_homework.ipynb`) is your **submission artifact**. This chapter tells you precisely what to fill in, what questions to answer, and how to package it.

---

## 9.1 The Homework Notebook's Actual Structure

Open `spec_dec+quantization_homework.ipynb`. It has exactly 10 cells:

| Cell | Type | Status | Action Required |
|---|---|---|---|
| cell-0 | markdown | Read-only | No changes |
| cell-1 | markdown | Task 1 desc + **1 question** | Add answer cell below it |
| cell-2 | markdown | Task 2 desc + **3 questions** | Add answer cells |
| cell-3 | markdown | Task 3 desc + **3 questions** | Add answer cells |
| cell-4 | markdown | Task 4 desc + **2 questions** + rubric | Add answer cells |
| cell-5 | markdown | Baseline example (already filled) | Read-only |
| **cell-6** | markdown | `TODO` — spec decoding results | **Replace TODO with your output** |
| **cell-7** | markdown | `TODO` — FP8 results | **Replace TODO with your output** |
| **cell-8** | markdown | `TODO` — FP8 + spec dec results | **Replace TODO with your output + explanation** |
| cell-9 | empty | Empty | Add your central question answer |

**Minimum to pass scoring:** Fill cells 6, 7, 8 with real benchmark output above the thresholds.  
**For full marks on combined (15 pts):** cell-8 must also include a draft-token tuning justification.

---

## 9.2 How to Capture Benchmark Output from Terminal

When you run `vllm bench serve`, the full output prints to terminal. Capture it like this:

```bash
# On your Nebius instance — pipe output to both terminal AND a file
vllm bench serve \
    --model Qwen/Qwen3-8B \
    --dataset-name hf \
    --dataset-path philschmid/mt-bench \
    --max-concurrency 8 \
    --num-prompts 80 \
    2>&1 | tee benchmark_spec_dec.txt

# Similarly for FP8
vllm bench serve \
    --model ./Qwen3-8B-FP8-Dynamic \
    --dataset-name hf \
    --dataset-path philschmid/mt-bench \
    --max-concurrency 8 \
    --num-prompts 80 \
    2>&1 | tee benchmark_fp8.txt

# And for FP8 + spec dec
vllm bench serve \
    --model ./Qwen3-8B-FP8-Dynamic \
    --dataset-name hf \
    --dataset-path philschmid/mt-bench \
    --max-concurrency 8 \
    --num-prompts 80 \
    2>&1 | tee benchmark_fp8_spec.txt
```

Then copy the content of each `.txt` file into the corresponding notebook cell.

---

## 9.3 Cell-by-Cell Fill Instructions

### cell-6: Speculative Decoding Results

Open the notebook. Find cell-6 which currently reads:
```
Speculative decoding benchmark results:
\`\`\`
TODO
\`\`\`
```

Replace `TODO` with the full terminal output from `benchmark_spec_dec.txt`. The final cell should look like:

```
Speculative decoding benchmark results:
\`\`\`
============ Serving Benchmark Result ============
Successful requests:                     80
...
Output token throughput (tok/s):         XXXX.XX    ← must be > 1250
...
==================================================
```

Also add a new markdown cell immediately after with your draft-token tuning note:
```
**Speculative decoding config:** num_speculative_tokens=N, chosen because...
```

---

### cell-7: FP8 Quantization Results

Find cell-7 (`TODO` — FP8 quantization benchmark results).  
Replace `TODO` with the full output from `benchmark_fp8.txt`.

```
FP8 quantization benchmark results:
\`\`\`
============ Serving Benchmark Result ============
...
Output token throughput (tok/s):         XXXX.XX    ← must be > 1550
...
==================================================
```

No explanation cell required here — this configuration has no tunable parameter.

---

### cell-8: FP8 + Speculative Decoding Results (needs explanation)

Find cell-8. Replace `TODO` with `benchmark_fp8_spec.txt` content.

**This cell needs more than just the output.** The rubric says "with draft-token tuning and explanation" for the 15-point combined result. Add a new markdown cell directly after cell-8 with this structure:

```markdown
### Combined (FP8 + Speculative Decoding) — Draft Token Tuning

**Draft tokens sweep results:**
| num_speculative_tokens | Output tok/s | Acceptance rate | Acceptance length |
|---|---|---|---|
| 1 | XXXX | XX% | X.XX |
| 2 | XXXX | XX% | X.XX |
| 3 | XXXX | XX% | X.XX |

**Chosen value:** `num_speculative_tokens = N`

**Justification:** At N draft tokens, throughput peaks at XXXX tok/s with acceptance
length X.XX. Increasing to N+1 draft tokens produced more drafts (XXXX) but lower
acceptance length (X.XX), resulting in lower net throughput — the overhead of rejected
drafts outweighed the benefit of additional accepted tokens. The FP8 verifier is
already faster than BF16, so the optimal operating point shifts to fewer draft tokens
compared to the BF16 + spec decoding configuration (which used M draft tokens).
```

---

### cell-9: Central Question Answer (Empty Cell)

cell-9 is currently empty. Add a markdown cell here with your answer to the central question:

```markdown
## Central Question: Which Should Be Done First?

**Answer: Quantization (FP8) should be applied first, before training the EAGLE-3 draft head.**

**Reasoning:**

The EAGLE-3 draft head is trained to predict the hidden states of a specific verifier model.
If we train on BF16 hidden states but serve with an FP8 verifier, the draft head encounters
a distribution shift — the FP8 model's hidden states differ slightly from BF16. This reduces
acceptance rate at serving time.

The correct pipeline is:
1. Quantize Qwen3-8B → Qwen3-8B-FP8-Dynamic
2. Generate hidden states using the FP8 model
3. Train EAGLE-3 draft head on FP8 hidden states
4. Serve: FP8 verifier + FP8-trained draft head

**Evidence from benchmark results:**
- Baseline:             XXX tok/s
- Spec decoding (BF16): XXX tok/s (+XX% — draft head trained on BF16 states)
- FP8 quantization:     XXX tok/s (+XX%)
- FP8 + spec decoding:  XXX tok/s (+XX% vs baseline)

The draft head in the combined run was trained on BF16 states but served with the FP8
verifier. Despite this distribution shift, the result still outperforms all other configs.
Training the draft head on FP8 states (optimal order) would yield even higher acceptance
rate and throughput than shown here.

**Practical benefit:** Generating hidden states from the FP8 verifier is also faster
(~1.5–2× speed improvement in the generation phase), reducing the total pipeline cost.
```

---

## 9.4 The Questions — Where to Answer Them

The task descriptions include questions that you should answer in the notebook. Add a markdown cell **directly after each task description cell**:

### After cell-1 (Task 1) — 1 Question

> Why do hidden states require much more disk space than the original text dataset?

```markdown
**Answer (Task 1):**

Text tokens are integers (~2 bytes each). Hidden states are float vectors: for Qwen3-8B,
each token position produces a vector of shape [4096] in BF16 = 8192 bytes.
EAGLE-3 uses hidden states from multiple layers, so the multiplier is (4096 × L × 2 bytes)
per token position × sequence length × number of samples.

For 3000 samples × 2048 tokens × 4096 dims × 3 layers × 2 bytes ≈ 147 GB,
compared to the text dataset which is well under 100 MB. The ratio is ~1500–2000×.
```

### After cell-2 (Task 2) — 3 Questions

```markdown
**Answers (Task 2):**

**Q1: What do `full_acc` and `cond_acc` measure?**
- `full_acc` (Full Accuracy): measures P(draft token k = verifier token k) when the draft
  head uses its own predictions for positions 0..k-1 as context. This is the real inference
  condition — errors in earlier positions cascade into later predictions.
- `cond_acc` (Conditional Accuracy): measures the same probability but using the ground-truth
  verifier tokens for positions 0..k-1 as context. This is the upper bound — what accuracy
  would be if earlier positions were always correct.

**Q2: Why does accuracy decrease for later speculative positions?**
Position k uses the draft head's prediction of position k-1 as context. If that prediction
is wrong (which happens at rate 1 - full_acc_{k-1}), the input to position k is corrupted,
making a correct prediction at position k even less likely. Errors compound multiplicatively
across positions.

**Q3: What would you change if first-position accuracy is very low (< 0.30)?**
First investigate data generation rather than training settings. If the vLLM server had
errors during hidden state generation, the hidden states may not correspond correctly to
the tokens in the dataset. Verify: (1) vLLM version is exactly 0.20.0, (2) no errors in
the generation logs, (3) sequence length alignment. Only if data is correct, then consider
increasing training epochs or adjusting learning rate.
```

### After cell-3 (Task 3) — 3 Questions

```markdown
**Answers (Task 3):**

**Q1: Why is FP8 dynamic quantization useful for serving on H100?**
LLM token generation is memory-bandwidth-bound. Each decode step loads all model weights
from HBM. Qwen3-8B in BF16 = 16 GB per step; in FP8 = 8 GB per step. The H100 has
3.35 TB/s HBM bandwidth, so FP8 doubles the effective bandwidth utilization per useful
computation. H100 also has native FP8 tensor cores that execute FP8 matrix multiplications
without precision conversion overhead.

**Q2: Why might `lm_head` be excluded from quantization?**
`lm_head` projects from hidden_dim (4096) to vocabulary size (152,064 for Qwen3). Its output
is the logit vector used for token probability. FP8 E4M3 has only ~1 decimal digit of
precision, which can cause the model to confuse tokens with similar logit values. Rare
tokens, formatting tokens, and disambiguation cases are particularly sensitive. Keeping
`lm_head` in BF16 costs only ~0.6 GB extra but preserves output quality.

**Q3: How can quantization affect speculative decoding acceptance rate?**
FP8 quantization changes the model's hidden state distribution slightly. A draft head
trained on BF16 hidden states and served against an FP8 verifier experiences a small
distribution shift. This means the hidden states the draft head predicts will not exactly
match what the FP8 verifier produces, potentially reducing acceptance rate. However,
in practice FP8 rounding can make the verifier's output distribution slightly sharper
(more concentrated probability mass on the top token), which can actually increase the
per-token acceptance rate while reducing acceptance length if fewer draft tokens are used.
```

### After cell-4 (Task 4) — 2 Questions

```markdown
**Answers (Task 4):**

**Q1: Why can speculative decoding improve throughput even when acceptance rate is < 100%?**
Acceptance rate (per-token) and throughput gain are not the same metric. The throughput
gain comes from producing more tokens per verifier forward pass (acceptance length), not
from perfect per-token acceptance.

Each verifier forward pass is dominated by memory bandwidth (loading ~16 GB of weights).
Verifying K draft tokens in parallel costs K× FLOPs but the same memory bandwidth as
verifying 1 — so additional verified tokens are nearly free in latency. Even if only
1.45 tokens are accepted per pass on average (from our 2-draft-token run), we produce
~45% more tokens per the same memory load than without spec decoding.

The break-even point is acceptance_length > 1.0 — and any positive acceptance rate
above ~0% achieves this (since we always get the corrected token even when all drafts
are rejected).

**Q2: How many speculative tokens are optimal, and how do you justify the choice?**
[Fill in from your draft token sweep results — see cell-8 explanation above]
For the BF16 + spec run: optimal was N draft tokens (acceptance length X.XX, throughput XXXX).
For the FP8 + spec run: optimal was M draft tokens (acceptance length Y.YY, throughput YYYY).
The FP8 verifier is faster per step, so the break-even point for wasted drafts shifts —
the cost of an unaccepted draft token is lower in absolute time, but so is the gain from
an accepted one. In practice, fewer draft tokens tend to be optimal for an already-fast
FP8 verifier.
```

---

## 9.5 Submission Packaging

Once all cells are filled, package and submit:

```bash
# On your Nebius instance
cd /data/hw3

# Verify the notebook is complete before zipping
jupyter nbconvert --to notebook --execute spec_dec+quantization_homework.ipynb \
    --ExecutePreprocessor.timeout=60 2>/dev/null \
    && echo "Notebook executes cleanly" \
    || echo "WARNING: notebook has execution errors (OK if cells are text-only)"

# Package — zip only the notebook, not the virtual environments
zip submission.zip spec_dec+quantization_homework.ipynb

# Verify contents
unzip -l submission.zip
# Expected output:
#   Archive:  submission.zip
#     Length      Date    Time    Name
#   ---------  ---------- -----   ----
#     XXXXXX  YYYY-MM-DD HH:MM   spec_dec+quantization_homework.ipynb
#   ---------                     -------
#     XXXXXX                      1 file

# Download to your local machine
# From your LOCAL terminal (not Nebius):
scp -i ~/.ssh/your_key ubuntu@YOUR_NEBIUS_IP:/data/hw3/submission.zip ./
```

**Do NOT include in the zip:**
- `speculators_venv/`, `vllm_venv/`, `comp_venv/` — explicitly excluded by instructions
- `hidden_states/` — too large
- `output/checkpoints/` — not required
- `Qwen3-8B-FP8-Dynamic/` — not required
- The `MASTERCLASS_GUIDE.ipynb` (that's your private learning journal)

---

## 9.6 Pre-Submission Checklist

Run through this before zipping:

```
[ ] cell-6: speculative decoding output pasted (not TODO)
    [ ] Output tok/s > 1250 (if not, re-tune num_speculative_tokens)

[ ] cell-7: FP8 quantization output pasted (not TODO)
    [ ] Output tok/s > 1550 (if not, re-run quantization and verify config.json)

[ ] cell-8: FP8 + spec decoding output pasted (not TODO)
    [ ] Output tok/s > 1750 (if not, try different num_speculative_tokens values)
    [ ] Draft token sweep table added
    [ ] Justification for chosen draft token count added

[ ] Answer cell added after cell-1 (hidden state disk space question)
[ ] Answer cell added after cell-2 (full_acc, cond_acc, low accuracy questions)
[ ] Answer cell added after cell-3 (FP8 usefulness, lm_head, effect on spec dec)
[ ] Answer cell added after cell-4 (throughput despite low rate, optimal token count)
[ ] cell-9: Central question answered (quantize first, with evidence)

[ ] Zip contains only the .ipynb file
[ ] Zip file name: submission.zip (or as specified by course)
```


---
# Appendix: Quick Reference

## Environment Activation Cheatsheet

```bash
# Data generation and training
source speculators_venv/bin/activate

# Serving and benchmarking
source vllm_venv/bin/activate

# FP8 quantization
source comp_venv/bin/activate
```

## Key Files and Directories

```
project/
├── speculators/              # Cloned speculators repo (v0.5.0)
├── hidden_states/            # Generated offline hidden states (~140 GB)
├── output/
│   └── checkpoints/          # EAGLE-3 draft head checkpoints
├── Qwen3-8B-FP8-Dynamic/     # Quantized verifier model
└── speculators_venv/
    vllm_venv/
    comp_venv/
```

## Live Training Monitoring

TensorBoard is installed automatically by `train_eagle3.sh` (`pip install -q tensorboard`).

```bash
# Terminal 1 — run training (inside tmux)
bash scripts/train_eagle3.sh

# Terminal 2 — stream val metrics
tail -f /data/hw3/output/train_run.log | grep -E "val/"

# Terminal 2 (alternative) — TensorBoard UI at http://localhost:6006
source /data/hw3/speculators_venv/bin/activate
tensorboard --logdir /data/hw3/logs --port 6006
```

**What to watch:**

| Metric | Signal |
|---|---|
| `val/loss_0_epoch` | Decreasing = learning; plateau = converged |
| `val/full_acc_0_epoch` | Target > 0.40; < 0.30 = problem upstream |
| `val/loss_epoch` | Total loss; reference ≈ 10.84 at epoch 4 |

Best checkpoint lives at `checkpoints/checkpoint_best/` (symlink set by `--save-best`).

```bash
ls -la /data/hw3/output/checkpoints/checkpoint_best
```

## Troubleshooting Reference

| Symptom | Likely Cause | Fix |
|---|---|---|
| Hidden state generation: missing temp files | Stale `/tmp/hidden_states/` files | `rm -rf /tmp/hidden_states/* && rerun` |
| Sequence length mismatch | Wrong vLLM version | Verify `vllm==0.20.0` exactly |
| Disk fills during generation | Too many samples | Reduce `--max-samples` before retry |
| Low position-0 accuracy (<0.30) | Bad data generation | Check vLLM server logs during gen |
| FP8 model config missing quantization section | Model not saved correctly | Re-run quantization script |
| Spec decoding slower than baseline | Too many draft tokens | Reduce `--num-speculative-tokens` |

## Key Metrics Glossary

| Metric | Meaning |
|---|---|
| Output tok/s | Primary throughput metric — output tokens generated per second |
| TTFT | Time to First Token — includes prefill (KV cache fill for the prompt) |
| TPOT | Time Per Output Token — decode latency per token, excludes first token |
| Acceptance rate | % of individual draft tokens accepted by the verifier |
| Acceptance length | Mean tokens accepted (+ 1 corrected) per draft-verify cycle |
| full_acc | Draft head accuracy using its own previous predictions as context |
| cond_acc | Draft head accuracy using ground-truth previous tokens as context |


---
# Appendix B — Documentation Navigation Guide

> When you hit a wall, read the right section — not the whole doc.

This guide maps each task in the assignment to the exact documentation section that will unblock you. All links are to stable versioned docs, not `latest`.

---

## B.1 Speculators Library (EAGLE-3 Training)

**Primary source:** [docs.vllm.ai/projects/speculators](https://docs.vllm.ai/projects/speculators/en/latest/)  
**GitHub:** [github.com/vllm-project/speculators](https://github.com/vllm-project/speculators) (tag `v0.5.0`)

| You need to... | Read this section |
|---|---|
| Understand the whole offline workflow | [User Guide → Tutorials → Train EAGLE-3 Offline](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline) |
| Understand what `launch_vllm.py` does | `speculators/scripts/launch_vllm.py` — read the argparse args, note `--port` and `--model` |
| Understand hidden state generation options | `speculators/scripts/data_generation_offline.py --help` (run it to see all flags) |
| Tune training hyperparameters | `speculators/scripts/train.py --help` and the training tutorial above |
| Understand `full_acc` vs `cond_acc` | [Concepts → EAGLE-3 Architecture](https://docs.vllm.ai/projects/speculators/en/latest/concepts/eagle3) — look for "conditional accuracy" |
| Troubleshoot stale temp files | [Troubleshooting section](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train_eagle3_offline#troubleshooting) of the offline training tutorial |

**What to read first (before writing any code):**  
The offline EAGLE-3 tutorial from top to bottom — it takes about 15 minutes and maps the entire pipeline. Do this before launching anything.

---

## B.2 vLLM (Serving & Benchmarking)

**Primary source:** [docs.vllm.ai](https://docs.vllm.ai/en/v0.8.0/) — use **v0.8.0** docs (closest to v0.20.0 behavior)  
**GitHub:** [github.com/vllm-project/vllm](https://github.com/vllm-project/vllm) (tag `v0.20.0`)

| You need to... | Read this section |
|---|---|
| Understand the vLLM server startup flags | [Docs → Serving → OpenAI-Compatible Server](https://docs.vllm.ai/en/latest/serving/openai_compatible_server.html) |
| Use speculative decoding with a draft head | [Docs → Inference → Speculative Decoding](https://docs.vllm.ai/en/latest/features/spec_decode.html) — look for "EAGLE" subsection |
| Disable prefix caching for fair benchmarks | Flag: `--no-enable-prefix-caching` on `vllm serve` |
| Run the benchmark tool | `vllm bench serve --help` — all options listed |
| Understand TTFT vs TPOT vs ITL | [Docs → Performance Metrics](https://docs.vllm.ai/en/latest/serving/metrics.html) |
| Load a quantized model | `vllm serve <path-to-fp8-model>` — vLLM auto-detects quantization from `config.json` |

**Critical flag to understand:**  
`--num-speculative-tokens` on `vllm serve` — this is what you tune in the draft token sweep.

---

## B.3 llmcompressor (FP8 Quantization)

**Primary source:** [github.com/vllm-project/llm-compressor](https://github.com/vllm-project/llm-compressor) (tag `v0.12.0`)

| You need to... | Read this section |
|---|---|
| See a complete FP8 dynamic quant example | [examples/quantization_w8a8_fp8/README.md](https://github.com/vllm-project/llm-compressor/blob/main/examples/quantization_w8a8_fp8/README.md) |
| Understand `QuantizationModifier` parameters | [src/llmcompressor/modifiers/quantization/README.md](https://github.com/vllm-project/llm-compressor/tree/main/src/llmcompressor/modifiers/quantization) |
| See all `scheme` options (FP8, INT8, etc.) | [examples/quantization_w8a8_int8/](https://github.com/vllm-project/llm-compressor/tree/main/examples/quantization_w8a8_int8) for comparison |
| Understand `ignore=["lm_head"]` effect | The README example explicitly shows this pattern — read the comments |
| Verify saved model structure | Check `config.json` → `quantization_config` section after saving |

**The entire FP8 dynamic quantization is ~20 lines of code.** If your implementation is much longer, you've overcomplicated it. Read the README example first.

---

## B.4 Research Papers — Read in This Order

These are the papers behind the techniques. Read them **after** you've run the experiments, not before. The code will make more sense when you've seen the results first.

| Paper | When to Read | What to Focus On |
|---|---|---|
| [Speculative Decoding (Leviathan et al., 2023)](https://arxiv.org/abs/2211.17192) | After baseline + C2 benchmarks | Section 3 (algorithm) and Table 1 (results) |
| [Fast Inference from Transformers via Speculative Decoding (Chen et al., 2023)](https://arxiv.org/abs/2302.01318) | Same time as above | Theorem 1 (lossless proof) — this is why output quality is preserved |
| [EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty (Li et al., 2024)](https://arxiv.org/abs/2401.15077) | After understanding basic spec decoding | Section 3 explains why hidden states beat token-level prediction |
| [EAGLE-2: Faster Inference of Language Models with Dynamic Draft Trees](https://arxiv.org/abs/2406.16858) | Optional | Dynamic tree construction — useful if you explore tree-based decoding |
| [FP8 Formats for Deep Learning (Micikevicius et al., 2022)](https://arxiv.org/abs/2209.05433) | Before or during quantization task | Sections 2-3: E4M3 vs E5M2 choice rationale |

---

## B.5 When You're Stuck — Diagnostic Decision Tree

```
Problem                          → Where to look
─────────────────────────────────────────────────────────────────
Gen fails: "missing temp files"  → rm -rf /tmp/hidden_states/*
Seq length mismatch              → vLLM version: must be 0.20.0 exactly
Disk filling too fast            → Reduce --max-samples by 50%, restart
Low position-0 accuracy < 0.30  → Re-run hidden state generation with logs
  (means draft head has no signal)   Check: does vLLM server log errors?
FP8 model won't load in vLLM    → Check config.json has quantization_config
  (quantization config missing)      Re-run quantize_qwen3.py
Spec decoding slower than base   → Draft tokens too high — start with 1, sweep up
Acceptance rate 0% (all rejected)→ Wrong --speculative-model path
  (draft model path issue)           Check: does the path contain config.json?
Training loss not decreasing     → Check hidden states exist and are non-empty
  (plateau from epoch 0)             du -sh ./hidden_states/
```

---

## B.6 Nebius-Specific Documentation

| Topic | Resource |
|---|---|
| Creating GPU instances | Nebius Console → Compute → Create Instance (UI is self-explanatory) |
| Attaching and formatting disks | [Nebius Docs → Compute → Disks](https://docs.nebius.com/compute/storage/create-disk) |
| S3-compatible object storage | [Nebius Docs → Object Storage](https://docs.nebius.com/object-storage/) |
| SSH key management | Console → Access → SSH Keys |
| Monitoring GPU utilization | `nvidia-smi dmon -s u` for live utilization, or use `nvtop` |

---

## B.7 Documentation Reading Strategy

> "Read enough to not be blocked, not enough to avoid thinking."

For each stage of the assignment:

1. **Before environment setup** — Read the offline EAGLE-3 tutorial top to bottom (15 min)
2. **Before hidden state generation** — Read `generate_hidden_states.py --help` output fully
3. **Before training** — Read `train.py --help` and scan the training script's docstring
4. **Before FP8 quantization** — Read the llmcompressor w8a8_fp8 README example completely
5. **Before benchmarking** — Read `vllm bench serve --help` and understand each metric name
6. **After getting results** — Read the speculative decoding paper (Leviathan et al.) to understand WHY your results look the way they do

**The goal is to have the documentation confirm what you already suspect from the experiments — not to use it as a substitute for running the experiments.**
